In [ ]:
import os
import psycopg
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), '.env'))

from analysis_notebooks.snowflake import connect

%config InlineBackend.figure_format = 'retina'


# Connect to Replica

In [ ]:
replica_conn_string = os.getenv("REPLICA_URL")
alignable_replica_conn = psycopg.connect(replica_conn_string)
alignable_replica_cursor = alignable_replica_conn.cursor()

# Connect to Snowflake

In [ ]:
alignable_reporting_conn = connect()
reporting_cursor = alignable_reporting_conn.cursor()

In [ ]:
def query(query_str):
	return reporting_cursor.execute(query_str).fetch_pandas_all()

# Edit Analysis

In [ ]:
query("""
    SELECT 
		g.DATA:source_user_id as source_user_id,
		g.DATA:target_user_id as target_user_id,
		g.DATA:message::STRING AS generated_message,
		g.DATA:message_type::STRING AS strategy,
		COALESCE(s.DATA:message::STRING, '') AS sent_message,
		s.DATA:edits_made::BOOLEAN AS edits_made,
		g.DATA:is_fallback::BOOLEAN AS is_fallback,
		g.DATA:interaction_location AS interaction_location,
		TO_VARCHAR(CONVERT_TIMEZONE('America/New_York', g.DATA:event_tstamp::TIMESTAMP_TZ), 'YYYY-MM-DD HH24:MI:SS') AS event_tstamp_est,
		p."prompt_content",
	FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
	LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
		ON g.DATA:tracking_id = s.DATA:tracking_id
		AND s.DATA:event_name = 'llm_message_sent'
	LEFT JOIN OPENFLOW_DB."public"."prompt_snapshots" p
		ON g.DATA:prompt_snapshot_id = p."id"
	WHERE g.DATA:event_name = 'llm_message_generated'
		AND s.DATA:edits_made::BOOLEAN = TRUE
	order by g.DATA:event_tstamp desc
""")

In [ ]:
query("""
WITH daily_edits AS (
    SELECT
        DATE(CONVERT_TIMEZONE('America/New_York', g.DATA:event_tstamp::TIMESTAMP_TZ)) AS event_date,
        COUNT(*) AS edit_count
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
        AND s.DATA:edits_made::BOOLEAN = TRUE
    GROUP BY 1
)
SELECT
    AVG(edit_count) AS avg_edits_per_day
FROM daily_edits		
""")

## Plot the number of edits made per day

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = query("""
    SELECT
        DATE(CONVERT_TIMEZONE('America/New_York', g.DATA:event_tstamp::TIMESTAMP_TZ)) AS event_date,
        COUNT(*) AS edit_count
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
        AND s.DATA:edits_made::BOOLEAN = TRUE
    GROUP BY 1
    ORDER BY 1
""")

df['EVENT_DATE'] = pd.to_datetime(df['EVENT_DATE'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['EVENT_DATE'], df['EDIT_COUNT'], marker='o', linewidth=2, markersize=4)
ax.set_title('Edits Made Per Day')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Edits')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
import difflib

df_edit_sizes = query("""
SELECT
    g.DATA:message::STRING AS generated_message,
    s.DATA:message::STRING AS sent_message
FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
INNER JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
    ON g.DATA:tracking_id = s.DATA:tracking_id
    AND s.DATA:event_name = 'llm_message_sent'
WHERE g.DATA:event_name = 'llm_message_generated'
""")

df_edit_sizes = df_edit_sizes[df_edit_sizes["GENERATED_MESSAGE"] != df_edit_sizes["SENT_MESSAGE"]].copy()

df_edit_sizes["pct_changed"] = df_edit_sizes.apply(
    lambda r: (1 - difflib.SequenceMatcher(
        None, r["GENERATED_MESSAGE"] or "", r["SENT_MESSAGE"] or ""
    ).ratio()) * 100,
    axis=1
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_edit_sizes["pct_changed"], bins=50, color="steelblue", edgecolor="white")
ax.set_xlabel("% of Message Changed")
ax.set_ylabel("Number of Edits")
ax.set_title("Distribution of Edit Magnitude")
ax.set_xlim(0, 100)
ax.axvline(df_edit_sizes["pct_changed"].median(), color="orange", linestyle="--", label=f'Median: {df_edit_sizes["pct_changed"].median():.1f}%')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
MIN_MESSAGES = 3  # exclude users with fewer messages than this

df_users = query(f"""
WITH user_edit_stats AS (
    SELECT
        g.DATA:source_user_id AS source_user_id,
        COUNT(*) AS total_messages,
        SUM(CASE WHEN s.DATA:edits_made::BOOLEAN = TRUE THEN 1 ELSE 0 END) AS total_edits,
        total_edits / total_messages AS edit_rate
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
    GROUP BY 1
    HAVING total_messages >= {MIN_MESSAGES}
)
SELECT
    source_user_id,
    total_messages,
    total_edits,
    ROUND(edit_rate * 100, 1) AS edit_rate_pct,
    CASE
        WHEN edit_rate = 0    THEN 'Never edits'
        WHEN edit_rate >= 0.7 THEN 'Almost always edits'
        ELSE                       'Sometimes edits'
    END AS edit_category
FROM user_edit_stats
ORDER BY edit_rate DESC
""")

category_order = ['Almost always edits', 'Sometimes edits', 'Never edits']
counts = df_users['EDIT_CATEGORY'].value_counts().reindex(category_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart: user counts per category
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title(f'Users by Edit Behavior (min {MIN_MESSAGES} messages)')
axes[0].set_ylabel('Number of Users')
for i, (cat, val) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, val + 0.3, str(val), ha='center', fontweight='bold')

# Histogram: distribution of individual edit rates
axes[1].hist(df_users['EDIT_RATE_PCT'], bins=20, color='steelblue', edgecolor='white')
axes[1].set_title('Distribution of User Edit Rates')
axes[1].set_xlabel('Edit Rate (%)')
axes[1].set_ylabel('Number of Users')
axes[1].axvline(70, color='#2ecc71', linestyle='--', label='Almost always (≥70%)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nTotal users analyzed: {len(df_users)}")
print(counts.to_string())


**Note:** Users with fewer than 3 messages are excluded — too few data points to reliably classify their edit behavior.

## Edit Theme Discovery

In [ ]:
import sys
import importlib
sys.path.insert(0, '/Users/garrettsmith/notebooks')

import helpers
importlib.reload(helpers)
from helpers import Helpers

h = Helpers(cache_dir="/Users/garrettsmith/notebooks/edit analysis/.llm_cache")  # uses ~/.aws/credentials automatically; pass region="us-west-2" if needed


In [ ]:
# Load all edits up to the cutoff (March 31, 2026 at noon ET)
EDIT_CUTOFF = '2026-03-31 12:00:00 -04:00'

df_edits = query(f"""
    SELECT
        g.DATA:source_user_id::STRING  AS source_user_id,
        g.DATA:message::STRING         AS generated_message,
        s.DATA:message::STRING         AS sent_message,
        g.DATA:interaction_location::STRING AS interaction_location
    FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
    LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
        ON g.DATA:tracking_id = s.DATA:tracking_id
        AND s.DATA:event_name = 'llm_message_sent'
    WHERE g.DATA:event_name = 'llm_message_generated'
        AND s.DATA:edits_made::BOOLEAN = TRUE
        AND s.DATA:event_tstamp::TIMESTAMP_TZ <= '{EDIT_CUTOFF}'::TIMESTAMP_TZ
""")

# Preprocess: normalize em-dashes, strip whitespace, drop blanks & identical pairs
df_edits = h.preprocess_edits(df_edits)
before = df_edits["GENERATED_MESSAGE"].tolist()
after  = df_edits["SENT_MESSAGE"].tolist()

print(f"{len(df_edits):,} edits to analyze")
print(f"Interaction locations: {df_edits['INTERACTION_LOCATION'].value_counts().to_dict()}")

In [ ]:
# Step 1: Estimate description cost before running (now uses Sonnet by default)
description_cost = h.estimate_description_cost(before, after)

In [ ]:
# Step 2: Generate structured annotations for each edit (Haiku + forced tool use)
annotations = h.describe_edits_parallel(before, after)

# Check for any annotations missing expected keys
expected_keys = {"edit_actions", "structural_change", "actionability_delta", "message_quality_impact", "intent"}
bad = [(i, set(a.keys())) for i, a in enumerate(annotations) if not expected_keys.issubset(a.keys())]
if bad:
    print(f"⚠ {len(bad)} annotation(s) missing keys:")
    for i, keys in bad[:5]:
        print(f"  [{i}] has keys: {keys}, missing: {expected_keys - keys}")
        print(f"       {annotations[i]}")
else:
    print(f"All {len(annotations)} annotations have expected keys")

# Unpack structured fields into the dataframe (with defaults for any missing keys)
df_edits["annotation"] = annotations
df_edits["intent"] = [a.get("intent", "") for a in annotations]
df_edits["edit_actions"] = [a.get("edit_actions", []) for a in annotations]
df_edits["structural_change"] = [a.get("structural_change", "") for a in annotations]
df_edits["actionability_delta"] = [a.get("actionability_delta", "neutral") for a in annotations]
df_edits["message_quality_impact"] = [a.get("message_quality_impact", "neutral") for a in annotations]

In [ ]:
# Step 3: Build embedding strings — concatenate structured fields + intent for richer signal
def build_embedding_text(a):
    actions = " | ".join(a.get("edit_actions", []))
    parts = [
        actions,
        a.get("actionability_delta", "neutral"),
        a.get("message_quality_impact", "neutral"),
        a.get("intent", ""),
    ]
    return " | ".join(p for p in parts if p)

embedding_texts = [build_embedding_text(a) for a in annotations]
embedding_cost = h.estimate_embedding_cost(embedding_texts)
embedding_texts[0]  # preview

In [ ]:
embedding_texts

In [ ]:
# Step 4: Embed concatenated structured fields + intent
embeddings = h.embed_texts(embedding_texts)

In [ ]:
# Cluster with 'leaf' method + low min_samples to reduce noise
labels = h.cluster_embeddings(
    embeddings,
    min_cluster_size=40,
    min_samples=5,
    umap_n_components=15,
    umap_n_neighbors=30,
    cluster_selection_method="eom",
)

In [ ]:
# Search for clustering params: ≤20 clusters, ≤30% noise after approximate_predict
# Loads from cached CSV if available, otherwise runs the search (~15-20 min)
import os

MAX_CLUSTERS = 20
MAX_NOISE_PCT = 0.30
csv_path = "/Users/garrettsmith/notebooks/edit analysis/.llm_cache/clustering_search_results.csv"

if os.path.exists(csv_path):
    print(f"Loading cached results from {csv_path}")
    df_all_results = pd.read_csv(csv_path)
    df_hits = df_all_results[
        (df_all_results["n_clusters"] <= MAX_CLUSTERS)
        & (df_all_results["noise_pct_after_approx"] <= MAX_NOISE_PCT)
    ].sort_values(["noise_pct_after_approx", "n_clusters"])
    print(f"{len(df_all_results):,} combinations loaded, {len(df_hits):,} meet criteria")
else:
    df_all_results, df_hits = h.search_clustering_params(
        embeddings,
        max_clusters=MAX_CLUSTERS,
        max_noise_pct=MAX_NOISE_PCT,
    )

print(f"\nTop 10 candidates:")
df_hits.head(10)

In [ ]:
# Filter to configs with 8-20 clusters
df_hits_8_20 = df_hits[(df_hits["n_clusters"] >= 8) & (df_hits["n_clusters"] <= 20)].sort_values(["noise_pct_after_approx", "n_clusters"])
print(f"{len(df_hits_8_20)} configs with 10-20 clusters:")
df_hits_8_20

In [ ]:
import re
import importlib; importlib.reload(helpers); from helpers import Helpers
h = Helpers(cache_dir="/Users/garrettsmith/notebooks/edit analysis/.llm_cache")

MIN_USERS_PER_CLUSTER = 10

# Run the full pipeline for each candidate in df_hits_8_20
candidate_results = {}

for idx, row in df_hits_8_20.iterrows():
    config_label = (f"umap({row['umap_n_components']:.0f}d, nn={row['umap_n_neighbors']:.0f}) "
                    f"hdbscan(mcs={row['min_cluster_size']:.0f}, ms={row['min_samples']:.0f}, {row['cluster_selection_method']})")
    print(f"\n{'='*80}")
    print(f"Config: {config_label}")
    print(f"Expected: {row['n_clusters']:.0f} clusters, {row['noise_pct_after_approx']:.1%} noise after approx_predict")
    print(f"{'='*80}")

    # Step 1: Cluster
    labels = h.cluster_embeddings(
        embeddings,
        min_cluster_size=int(row["min_cluster_size"]),
        min_samples=int(row["min_samples"]),
        umap_n_components=int(row["umap_n_components"]),
        umap_n_neighbors=int(row["umap_n_neighbors"]),
        cluster_selection_method=row["cluster_selection_method"],
    )

    # Step 2: Filter small clusters by unique users
    df_edits["cluster_id"] = labels
    user_counts = df_edits.groupby("cluster_id")["SOURCE_USER_ID"].nunique()
    small_clusters = user_counts[user_counts < MIN_USERS_PER_CLUSTER].index.tolist()
    df_edits.loc[df_edits["cluster_id"].isin(small_clusters), "cluster_id"] = -1
    labels = df_edits["cluster_id"].values

    n_clusters = df_edits[df_edits["cluster_id"] != -1]["cluster_id"].nunique()
    n_noise = (df_edits["cluster_id"] == -1).sum()
    print(f"After user filter: {n_clusters} clusters, {n_noise} noise points ({n_noise / len(df_edits):.1%} uncategorized)")

    # Step 3: Label clusters
    themes, reasonings, labeling_cost = h.label_all_clusters(annotations, labels)
    themes = {k: re.sub(r"[#*]+", "", v.split("\n")[0]).strip() for k, v in themes.items()}

    # Step 4: Consolidate
    proposed_groups = h.propose_consolidation(themes)
    themes = h.apply_consolidation(themes, proposed_groups)

    # Step 5: Map themes and plot
    df_edits["theme"] = df_edits["cluster_id"].map(themes)
    df_edits["theme_reasoning"] = df_edits["cluster_id"].map(reasonings)

    theme_counts = (
        df_edits[df_edits["cluster_id"] != -1]
        .groupby("theme")
        .size()
        .sort_values(ascending=False)
        .reset_index(name="edit_count")
    )
    total = theme_counts["edit_count"].sum()

    fig, ax = plt.subplots(figsize=(10, max(6, len(theme_counts) * 0.55)))
    bars = ax.barh(theme_counts["theme"], theme_counts["edit_count"], color="steelblue")
    ax.invert_yaxis()
    ax.set_xlabel("Number of Edits")
    ax.set_title(f"Edit Themes by Frequency — {config_label}")
    ax.tick_params(axis="y", labelsize=8)
    for bar, count in zip(bars, theme_counts["edit_count"]):
        ax.text(bar.get_width() + ax.get_xlim()[1] * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{count} ({count / total:.1%})", va="center", ha="left", fontsize=8)
    ax.margins(x=0.15)
    plt.tight_layout()
    plt.show()

    candidate_results[config_label] = {
        "labels": labels.copy(),
        "themes": themes,
        "reasonings": reasonings,
        "theme_counts": theme_counts,
    }

print(f"\n{'='*80}")
print(f"Done — {len(candidate_results)} configs evaluated. Results stored in candidate_results dict.")

In [ ]:
import numpy as np
import json

# Persist the chosen config: umap(30d, nn=50) hdbscan(mcs=100, ms=15, eom)
chosen_key = [k for k in candidate_results if "mcs=100, ms=15, eom" in k][0]
chosen = candidate_results[chosen_key]

persist_dir = "/Users/garrettsmith/notebooks/edit analysis/.llm_cache"
np.save(f"{persist_dir}/chosen_labels.npy", chosen["labels"])
with open(f"{persist_dir}/chosen_themes.json", "w") as f:
    json.dump({str(k): v for k, v in chosen["themes"].items()}, f, indent=2)
with open(f"{persist_dir}/chosen_reasonings.json", "w") as f:
    json.dump({str(k): v for k, v in chosen["reasonings"].items()}, f, indent=2)

print(f"Saved: {chosen_key}")
print(f"  labels:     {persist_dir}/chosen_labels.npy")
print(f"  themes:     {persist_dir}/chosen_themes.json")
print(f"  reasonings: {persist_dir}/chosen_reasonings.json")

In [ ]:
# Load persisted clustering result (or fall through to recompute below)
import json as _json

persist_dir = "/Users/garrettsmith/notebooks/edit analysis/.llm_cache"
labels = np.load(f"{persist_dir}/chosen_labels.npy")
with open(f"{persist_dir}/chosen_themes.json") as f:
    themes = {int(k): v for k, v in _json.load(f).items()}
with open(f"{persist_dir}/chosen_reasonings.json") as f:
    reasonings = {int(k): v for k, v in _json.load(f).items()}

df_edits["cluster_id"] = labels
n_clusters = df_edits[df_edits["cluster_id"] != -1]["cluster_id"].nunique()
n_noise = (df_edits["cluster_id"] == -1).sum()
print(f"Loaded: {n_clusters} clusters, {n_noise} noise points ({n_noise / len(df_edits):.1%} uncategorized)")
print(f"Config: umap(30d, nn=50) hdbscan(mcs=100, ms=15, eom)")

In [ ]:
# Step 5.5: Estimate labeling cost before running (uses Opus)
labeling_cost_estimate = h.estimate_labeling_cost(annotations, labels)

In [ ]:
import re

# Step 6: Label each cluster with a theme + reasoning (now uses structured annotations for richer context)
themes, reasonings, labeling_cost = h.label_all_clusters(annotations, labels)

In [ ]:
# Collapse to first line, strip markdown formatting and whitespace
themes = {k: re.sub(r"[#*]+", "", v.split("\n")[0]).strip() for k, v in themes.items()}

In [ ]:
import importlib; importlib.reload(helpers); from helpers import Helpers; h = Helpers(cache_dir="/Users/garrettsmith/notebooks/edit analysis/.llm_cache")

# Step 1: Propose consolidation — review before applying
proposed_groups = h.propose_consolidation(themes)

In [ ]:
# Step 2: Apply consolidation (edit proposed_groups above if needed before running this)
themes = h.apply_consolidation(themes, proposed_groups)

# Show every canonical label and how many clusters mapped to it
from collections import Counter
canonical_counts = Counter(v for k, v in themes.items() if k != -1)
merged = {label: count for label, count in canonical_counts.items() if count > 1}
if merged:
    print("\nMerged groups (canonical label → clusters collapsed):")
    for label, count in sorted(merged.items(), key=lambda x: -x[1]):
        print(f"  {count}x  {label}")

In [ ]:
df_edits["theme"] = df_edits["cluster_id"].map(themes)
df_edits["theme_reasoning"] = df_edits["cluster_id"].map(reasonings)

def plot_theme_frequency():
    theme_counts = (
        df_edits[df_edits["cluster_id"] != -1]
        .groupby("theme").size().sort_values(ascending=False).reset_index(name="edit_count")
    )
    total = theme_counts["edit_count"].sum()
    fig, ax = plt.subplots(figsize=(10, max(6, len(theme_counts) * 0.55)))
    bars = ax.barh(theme_counts["theme"], theme_counts["edit_count"], color="steelblue")
    ax.invert_yaxis()
    ax.set_xlabel("Number of Edits")
    ax.set_title("Edit Themes by Frequency")
    ax.tick_params(axis="y", labelsize=8)
    for bar, count in zip(bars, theme_counts["edit_count"]):
        ax.text(bar.get_width() + ax.get_xlim()[1] * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{count} ({count / total:.1%})", va="center", ha="left", fontsize=8)
    ax.margins(x=0.15)
    fig.tight_layout()
    return fig, theme_counts

fig, theme_counts = plot_theme_frequency()
plt.show()
theme_counts

## Edit Themes by Interaction Location

In [ ]:
df_edits.head(1)

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

# Build contingency table: rows = interaction location, columns = theme
# Exclude noise cluster and null locations
df_located = df_edits[(df_edits["cluster_id"] != -1) & df_edits["INTERACTION_LOCATION"].notna()].copy()

contingency = (
    df_located
    .groupby(["INTERACTION_LOCATION", "theme"])
    .size()
    .unstack(fill_value=0)
)

# --- Chi-square test ---
chi2, p, dof, _ = chi2_contingency(contingency)

# Cramér's V
n      = contingency.values.sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"Chi-square: {chi2:.2f}  |  dof: {dof}  |  p-value: {p:.2e}")
print(f"Cramér's V: {cramers_v:.3f}  (0 = no association, 1 = perfect association)")
print()
if p < 0.05:
    print("✓ Edit theme distribution differs significantly across interaction locations (p < 0.05).")
else:
    print("✗ No statistically significant difference across interaction locations (p ≥ 0.05).")


In [ ]:
LOCATION_LABELS = {
    "connection_accept_llm_modal": "Post-Connection Accept Modal",
    "connection_request_introduce_yourself_llm_modal": "Post-Connection Request Modal",
    "daily_action_plan_introduce_yourself_llm": "Daily Action Plan Post-Connection Request Screen",
    "daily_action_plan_introduce_yourself_llm_modal": "Daily Action Plan Post-Connection Request Screen",
    "dap_accept_connection_message": "Daily Action Plan Post-Connection Accept Screen",
    "inbox_write_for_me": "Write For Me in Messaging",
}

def plot_theme_by_location():
    df_located = df_edits[(df_edits["cluster_id"] != -1) & df_edits["INTERACTION_LOCATION"].notna()].copy()
    contingency = pd.crosstab(df_located["INTERACTION_LOCATION"], df_located["theme"])
    normalised = contingency.div(contingency.sum(axis=1), axis=0) * 100
    normalised.index = [LOCATION_LABELS.get(loc, loc) for loc in normalised.index]

    n_themes = len(normalised.columns); n_locations = len(normalised.index)
    fig, ax = plt.subplots(figsize=(max(16, n_themes * 1.1), max(5, n_locations * 1.2)))
    im = ax.imshow(normalised.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="% of edits in location")
    ax.set_xticks(range(n_themes)); ax.set_xticklabels(normalised.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(n_locations)); ax.set_yticklabels(normalised.index, fontsize=9)
    ax.set_title("Edit Theme Distribution by Interaction Location (row-normalised %)")
    for i in range(n_locations):
        for j in range(n_themes):
            val = normalised.values[i, j]
            if val > 0:
                ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7,
                        color="black" if val < normalised.values.max() * 0.6 else "white")
    fig.tight_layout()
    fig.subplots_adjust(left=max(0.15, max(len(s) for s in normalised.index) * 0.008))
    return fig

fig = plot_theme_by_location()
plt.show()

## 1. Edit Action Frequency — What Does the AI Consistently Get Wrong?
Horizontal bar chart of every action tag, ranked by how often it appears across all edits.

In [ ]:
def plot_action_frequency():
    all_actions = df_edits[df_edits["cluster_id"] != -1].explode("edit_actions")
    action_counts = all_actions["edit_actions"].value_counts()
    total_edits = df_edits[df_edits["cluster_id"] != -1].shape[0]
    action_counts_filtered = action_counts[action_counts / total_edits > 0.10]

    fig, ax = plt.subplots(figsize=(12, max(6, len(action_counts_filtered) * 0.35)))
    bars = ax.barh(action_counts_filtered.index, action_counts_filtered.values, color="steelblue")
    ax.invert_yaxis(); ax.set_xlabel("Number of Edits")
    ax.set_title("Edit Action Frequency — actions in >10% of edits")
    ax.tick_params(axis="y", labelsize=8)
    for bar, count in zip(bars, action_counts_filtered.values):
        ax.text(bar.get_width() + total_edits * 0.005, bar.get_y() + bar.get_height() / 2,
                f"{count:,} ({count / total_edits:.1%})", va="center", ha="left", fontsize=7)
    ax.margins(x=0.15); fig.tight_layout()
    return fig, action_counts

fig, action_counts = plot_action_frequency()
plt.show()
action_counts.head(20)

## 2. Actionability x Quality Impact Crosstab
Heatmap showing whether users are making messages less pushy-but-better or less pushy-and-worse.

In [ ]:
def plot_actionability_vs_quality():
    df_clustered = df_edits[df_edits["cluster_id"] != -1]
    ct = pd.crosstab(df_clustered["actionability_delta"], df_clustered["message_quality_impact"], margins=True)
    action_order = ["much_more_actionable", "slightly_more_actionable", "neutral", "slightly_less_actionable", "much_less_actionable", "All"]
    quality_order = ["improved_toward_response_worthy", "neutral", "degraded_toward_not_response_worthy", "All"]
    ct = ct.reindex(index=[a for a in action_order if a in ct.index], columns=[q for q in quality_order if q in ct.columns])
    ct_pct = (ct / len(df_clustered) * 100).round(1)
    ct_heat = ct_pct.iloc[:-1, :-1]

    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(ct_heat.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="% of all edits")
    ax.set_xticks(range(len(ct_heat.columns))); ax.set_xticklabels(ct_heat.columns, rotation=30, ha="right", fontsize=9)
    ax.set_yticks(range(len(ct_heat.index))); ax.set_yticklabels(ct_heat.index, fontsize=9)
    ax.set_title("Actionability Delta vs Message Quality Impact (% of edits)")
    for i in range(len(ct_heat.index)):
        for j in range(len(ct_heat.columns)):
            val = ct_heat.values[i, j]
            if val > 0:
                ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=9,
                        color="black" if val < ct_heat.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig, ct

fig, ct = plot_actionability_vs_quality()
plt.show()
print("\nRaw counts:")
ct

## 3. Edit Patterns by Interaction Location
Heatmap of top 15 actions broken down by interaction location. Reveals if different contexts need different prompts.

In [ ]:
def plot_actions_by_location():
    df_loc = df_edits[df_edits["cluster_id"] != -1].copy()
    df_loc["location_label"] = df_loc["INTERACTION_LOCATION"].map(LOCATION_LABELS).fillna(df_loc["INTERACTION_LOCATION"])
    df_loc_exploded = df_loc.explode("edit_actions")

    loc_action_ct = pd.crosstab(df_loc_exploded["location_label"], df_loc_exploded["edit_actions"])
    loc_totals = df_loc.groupby("location_label").size()
    loc_action_pct = loc_action_ct.div(loc_totals, axis=0) * 100

    top_actions = df_loc_exploded["edit_actions"].value_counts().head(15).index.tolist()
    loc_action_pct_top = loc_action_pct.reindex(columns=top_actions)

    fig, ax = plt.subplots(figsize=(16, max(4, len(loc_action_pct_top) * 0.8)))
    im = ax.imshow(loc_action_pct_top.values, aspect="auto", cmap="YlGnBu")
    plt.colorbar(im, ax=ax, label="% of edits in location with this action")
    ax.set_xticks(range(len(top_actions))); ax.set_xticklabels(top_actions, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(loc_action_pct_top.index))); ax.set_yticklabels(loc_action_pct_top.index, fontsize=9)
    ax.set_title("Top Edit Actions by Interaction Location (% of edits per location)")
    for i in range(len(loc_action_pct_top.index)):
        for j in range(len(top_actions)):
            val = loc_action_pct_top.values[i, j]
            if val > 0:
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7,
                        color="black" if val < loc_action_pct_top.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig

fig = plot_actions_by_location()
plt.show()

## 4. Repeat Editor Analysis — Per-User Edit Consistency
For users with 5+ edits, how consistent are their edits? Histogram of theme concentration and most common dominant themes.

In [ ]:
def plot_repeat_editors():
    MIN_EDITS = 5
    df_clustered = df_edits[df_edits["cluster_id"] != -1].copy()
    user_edit_counts = df_clustered.groupby("SOURCE_USER_ID").size()
    heavy_editors = user_edit_counts[user_edit_counts >= MIN_EDITS].index
    df_heavy = df_clustered[df_clustered["SOURCE_USER_ID"].isin(heavy_editors)].copy()

    user_theme_stats = []
    for uid, group in df_heavy.groupby("SOURCE_USER_ID"):
        tc = group["theme"].value_counts()
        user_theme_stats.append({
            "user_id": uid, "n_edits": len(group), "top_theme": tc.index[0],
            "top_theme_pct": tc.iloc[0] / len(group), "n_distinct_themes": len(tc),
        })
    df_user_stats = pd.DataFrame(user_theme_stats)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_user_stats["top_theme_pct"], bins=20, color="steelblue", edgecolor="white")
    axes[0].set_xlabel("% of edits in user's dominant theme"); axes[0].set_ylabel("Number of users")
    axes[0].set_title(f"Edit Consistency Among Heavy Editors (n={len(df_user_stats):,})")
    axes[0].axvline(df_user_stats["top_theme_pct"].median(), color="red", linestyle="--",
                    label=f"Median: {df_user_stats['top_theme_pct'].median():.0%}")
    axes[0].legend()
    axes[1].hist(df_user_stats["n_distinct_themes"], bins=range(1, df_user_stats["n_distinct_themes"].max() + 2),
                 color="coral", edgecolor="white")
    axes[1].set_xlabel("Number of distinct themes per user"); axes[1].set_ylabel("Number of users")
    axes[1].set_title("Theme Diversity Per Heavy Editor")
    fig.tight_layout()
    return fig, df_user_stats

fig, df_user_stats = plot_repeat_editors()
plt.show()
print("\nMost common dominant themes among heavy editors:")
df_user_stats["top_theme"].value_counts().head(10)

## 5. Edit Action Co-occurrence
Co-occurrence matrix and top pairs. Reveals compound behaviors (e.g., `removed_cta` + `shortened_message` always appearing together).

In [ ]:
from itertools import combinations

def plot_action_cooccurrence():
    df_clustered = df_edits[df_edits["cluster_id"] != -1]
    pair_counts = Counter()
    for actions in df_clustered["edit_actions"]:
        if isinstance(actions, list) and len(actions) >= 2:
            for pair in combinations(sorted(set(actions)), 2):
                pair_counts[pair] += 1

    top_n = 15
    top_actions = df_clustered.explode("edit_actions")["edit_actions"].value_counts().head(top_n).index.tolist()
    cooccurrence = pd.DataFrame(0, index=top_actions, columns=top_actions)
    for (a1, a2), count in pair_counts.items():
        if a1 in top_actions and a2 in top_actions:
            cooccurrence.loc[a1, a2] = count; cooccurrence.loc[a2, a1] = count

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cooccurrence.values, aspect="auto", cmap="Blues")
    plt.colorbar(im, ax=ax, label="Co-occurrence count")
    ax.set_xticks(range(top_n)); ax.set_xticklabels(top_actions, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(top_n)); ax.set_yticklabels(top_actions, fontsize=8)
    ax.set_title("Edit Action Co-occurrence (top 15 actions)")
    for i in range(top_n):
        for j in range(top_n):
            val = cooccurrence.values[i, j]
            if val > 0 and i != j:
                ax.text(j, i, f"{val:,}", ha="center", va="center", fontsize=6,
                        color="black" if val < cooccurrence.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig, pair_counts

fig, pair_counts = plot_action_cooccurrence()
plt.show()
print("Top 10 co-occurring action pairs:")
for (a1, a2), count in pair_counts.most_common(10):
    print(f"  {count:>5,}x  {a1}  +  {a2}")

## 6. "Gutted Message" Analysis — Edits That Discard Most of the AI Output
Filters to edits where users threw away most of the output. Breaks down by action, theme, and interaction location.

In [ ]:
# Edits where the user gutted the message: converted to simple greeting, much_less_actionable, or kept < 30% of original
df_clustered = df_edits[df_edits["cluster_id"] != -1].copy()

# Compute retention ratio (sent length / generated length)
df_clustered["retention_ratio"] = df_clustered["SENT_MESSAGE"].str.len() / df_clustered["GENERATED_MESSAGE"].str.len()

gutted_mask = (
    df_clustered["edit_actions"].apply(lambda a: "converted_to_simple_greeting" in a or "converted_to_thank_you" in a)
    | (df_clustered["actionability_delta"] == "much_less_actionable")
    | (df_clustered["retention_ratio"] < 0.3)
)
df_gutted = df_clustered[gutted_mask]

print(f"Gutted messages: {len(df_gutted):,} ({len(df_gutted)/len(df_clustered):.1%} of edits)")
print(f"\nRetention ratio stats (sent_len / generated_len):")
print(df_gutted["retention_ratio"].describe().to_string())

# What actions appear most in gutted messages?
gutted_actions = df_gutted.explode("edit_actions")["edit_actions"].value_counts().head(10)
print(f"\nTop actions in gutted messages:")
for action, count in gutted_actions.items():
    print(f"  {count:>5,}x ({count/len(df_gutted):.0%})  {action}")

# Which themes are most common among gutted messages?
print(f"\nThemes of gutted messages:")
gutted_themes = df_gutted["theme"].value_counts()
for theme, count in gutted_themes.items():
    print(f"  {count:>5,}x ({count/len(df_gutted):.0%})  {theme}")

# Which interaction locations have the highest gutting rate?
print(f"\nGutting rate by interaction location:")
loc_labels = {
    "connection_accept_llm_modal": "Post-Connection Accept Modal",
    "connection_request_introduce_yourself_llm_modal": "Post-Connection Request Modal",
    "daily_action_plan_introduce_yourself_llm_modal": "DAP Post-Connection Request",
    "dap_accept_connection_message": "DAP Post-Connection Accept",
    "inbox_write_for_me": "Write For Me in Messaging",
}
for loc, group in df_clustered.groupby("INTERACTION_LOCATION"):
    loc_gutted = group[gutted_mask.reindex(group.index, fill_value=False)]
    label = loc_labels.get(loc, loc)
    print(f"  {label}: {len(loc_gutted):,}/{len(group):,} ({len(loc_gutted)/len(group):.1%})")

## 7. Edits That Improve Messages — What Are Users Adding That the AI Missed?
Filters to edits that improved message quality, computes lift of each action vs baseline to find what users add that the AI misses.

In [ ]:
# Edits that improved the message toward being response-worthy
df_clustered = df_edits[df_edits["cluster_id"] != -1].copy()
df_improved = df_clustered[df_clustered["message_quality_impact"] == "improved_toward_response_worthy"]

print(f"Improved edits: {len(df_improved):,} ({len(df_improved)/len(df_clustered):.1%} of all edits)")

# What actions do improving edits take?
improved_actions = df_improved.explode("edit_actions")["edit_actions"].value_counts()
all_actions = df_clustered.explode("edit_actions")["edit_actions"].value_counts()

# Compare: over-represented actions in improved edits vs all edits
action_comparison = pd.DataFrame({
    "improved_pct": (improved_actions / len(df_improved) * 100).round(1),
    "overall_pct": (all_actions / len(df_clustered) * 100).round(1),
})
action_comparison["lift"] = (action_comparison["improved_pct"] / action_comparison["overall_pct"]).round(2)
action_comparison = action_comparison.dropna().sort_values("lift", ascending=False)

print("\nActions over-represented in improving edits (lift > 1 = more common in improvements):")
for action, row in action_comparison.head(15).iterrows():
    print(f"  {action:<45} {row['improved_pct']:>5.1f}% in improved vs {row['overall_pct']:>5.1f}% overall  (lift: {row['lift']:.1f}x)")

# Actionability distribution of improving edits
print(f"\nActionability delta for improving edits:")
for delta, count in df_improved["actionability_delta"].value_counts().items():
    print(f"  {delta}: {count:,} ({count/len(df_improved):.0%})")

# Themes of improving edits
print(f"\nThemes of improving edits:")
for theme, count in df_improved["theme"].value_counts().head(10).items():
    print(f"  {count:>5,}x ({count/len(df_improved):.0%})  {theme}")

## Making a spreadsheet to show edits

In [ ]:
edits_by_theme = (
    df_edits[df_edits["cluster_id"] != -1][["theme", "theme_reasoning", "intent", "edit_actions", "actionability_delta", "message_quality_impact", "GENERATED_MESSAGE", "SENT_MESSAGE"]]
    .assign(_theme_count=lambda d: d.groupby("theme")["theme"].transform("count"))
    .sort_values(["_theme_count", "theme"], ascending=[False, True])
    .drop(columns="_theme_count")
    .reset_index(drop=True)
)

edits_by_theme

In [ ]:
import difflib
from openpyxl import Workbook
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font

def make_rich_diff(original: str, edited: str) -> CellRichText:
    """Word-level diff as openpyxl CellRichText.
    Deletions = red strikethrough, insertions = green bold, unchanged = black."""
    orig_words = original.split()
    edit_words = edited.split()

    font_normal  = InlineFont(color="000000")
    font_deleted = InlineFont(color="CC0000", strike=True)
    font_added   = InlineFont(color="007700", b=True)

    blocks = []
    matcher = difflib.SequenceMatcher(None, orig_words, edit_words, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            blocks.append(TextBlock(font_normal,  " ".join(orig_words[i1:i2]) + " "))
        elif op == "delete":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
        elif op == "insert":
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))
        elif op == "replace":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))

    return CellRichText(*blocks) if blocks else CellRichText("")


wb = Workbook()
ws = wb.active
ws.title = "Edit Themes"

# Header row
headers = ["Theme", "Theme Reasoning", "Intent", "Edit Actions", "Actionability Delta",
           "Message Quality Impact", "Generated Message", "Sent Message (Annotated)"]
for col, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col, value=header)
    cell.font = Font(bold=True)

# Column widths
ws.column_dimensions["A"].width = 30
ws.column_dimensions["B"].width = 50
ws.column_dimensions["C"].width = 40
ws.column_dimensions["D"].width = 40
ws.column_dimensions["E"].width = 25
ws.column_dimensions["F"].width = 30
ws.column_dimensions["G"].width = 60
ws.column_dimensions["H"].width = 60

wrap_top = Alignment(wrap_text=True, vertical="top")

# Data rows
for row_idx, row in enumerate(edits_by_theme.itertuples(), start=2):
    ws.cell(row=row_idx, column=1, value=row.theme).alignment = wrap_top
    ws.cell(row=row_idx, column=2, value=row.theme_reasoning).alignment = wrap_top
    ws.cell(row=row_idx, column=3, value=row.intent).alignment = wrap_top
    ws.cell(row=row_idx, column=4, value=", ".join(row.edit_actions) if isinstance(row.edit_actions, list) else str(row.edit_actions)).alignment = wrap_top
    ws.cell(row=row_idx, column=5, value=row.actionability_delta).alignment = wrap_top
    ws.cell(row=row_idx, column=6, value=row.message_quality_impact).alignment = wrap_top
    ws.cell(row=row_idx, column=7, value=row.GENERATED_MESSAGE).alignment = wrap_top

    diff_cell = ws.cell(row=row_idx, column=8)
    diff_cell.value     = make_rich_diff(row.GENERATED_MESSAGE or "", row.SENT_MESSAGE or "")
    diff_cell.alignment = wrap_top

output_path = "/Users/garrettsmith/notebooks/edit analysis/edit_themes.xlsx"
wb.save(output_path)
print(f"Saved → {output_path}")

# Contextual Analysis

Let's pull in additional context by which to analyse these edits. 
- lifecycle stage before the edited message was sent
- strategy used to generate the message
	- available in the query below: df_context_analysis["STRATEGY"]
- specific prompt/variation used to generate the message
	- available in the query below: df_context_analysis["DATA"] (json) in the "prompt_snapshot_id" property
- contextual data between users 
	- available in the query below: df_context_analysis["DATA"] (json)


In [ ]:
from IPython.display import display, HTML

def preview_long_string(s, max_height="200px"):
    escaped = s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML(f"""
        <div style="
            max-height: {max_height};
            overflow-y: auto;
            white-space: pre-wrap;
            word-wrap: break-word;
            font-family: monospace;
            padding: 8px;
            border: 1px solid #ccc;
            border-radius: 4px;
        ">{escaped}</div>
    """))

In [ ]:
df_context_analysis = query("""
    SELECT 
		g.DATA:source_user_id as source_user_id,
		g.DATA:target_user_id as target_user_id,
		g.DATA:message::STRING AS generated_message,
		g.DATA:message_type::STRING AS strategy,
		COALESCE(s.DATA:message::STRING, '') AS sent_message,
		s.DATA:edits_made::BOOLEAN AS edits_made,
		g.DATA:is_fallback::BOOLEAN AS is_fallback,
		g.DATA:interaction_location AS interaction_location,
		TO_VARCHAR(CONVERT_TIMEZONE('America/New_York', g.DATA:event_tstamp::TIMESTAMP_TZ), 'YYYY-MM-DD HH24:MI:SS') AS event_tstamp_est,
		p."prompt_content",
		g.*
	FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
	LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
		ON g.DATA:tracking_id = s.DATA:tracking_id
		AND s.DATA:event_name = 'llm_message_sent'
	LEFT JOIN OPENFLOW_DB."public"."prompt_snapshots" p
		ON g.DATA:prompt_snapshot_id = p."id"
	WHERE g.DATA:event_name = 'llm_message_generated'
	order by g.DATA:event_tstamp desc
""")

In [ ]:
df_context_analysis.head(1)

In [ ]:
df_context_analysis = query("""
    SELECT 
		g.DATA:source_user_id as source_user_id,
		g.DATA:target_user_id as target_user_id,
		g.DATA:message::STRING AS generated_message,
		g.DATA:message_type::STRING AS strategy,
		COALESCE(s.DATA:message::STRING, '') AS sent_message,
		s.DATA:edits_made::BOOLEAN AS edits_made,
		g.DATA:is_fallback::BOOLEAN AS is_fallback,
		g.DATA:interaction_location AS interaction_location,
		TO_VARCHAR(CONVERT_TIMEZONE('America/New_York', g.DATA:event_tstamp::TIMESTAMP_TZ), 'YYYY-MM-DD HH24:MI:SS') AS event_tstamp_est,
		p."prompt_content",
		g.DATA
	FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
	LEFT JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
		ON g.DATA:tracking_id = s.DATA:tracking_id
		AND s.DATA:event_name = 'llm_message_sent'
	LEFT JOIN OPENFLOW_DB."public"."prompt_snapshots" p
		ON g.DATA:prompt_snapshot_id = p."id"
	WHERE g.DATA:event_name = 'llm_message_generated'
	order by g.DATA:event_tstamp desc
""")

In [ ]:
import json

data = df_context_analysis["DATA"].head(1).item()
json.loads(data)

In [ ]:
json.loads(df_context_analysis["DATA"].head(1).item())["context"]["state"]["prompt_variant"]

In [ ]:
# Extract relationship_data from each row into a new dataframe
df_relationship = pd.json_normalize(
    df_context_analysis["DATA"]
    .apply(lambda d: json.loads(d).get("context", {}).get("state", {}).get("prompt_context", {}).get("input", {}).get("relationship_data", {}))
)
df_relationship.index = df_context_analysis.index
df_relationship

In [ ]:
df_relationship.columns

In [ ]:
# One example of each column where the value is non-None
examples = {}
for col in df_relationship.columns:
    non_null = df_relationship[col].dropna()
    if len(non_null) > 0:
        examples[col] = non_null.iloc[0]

pd.DataFrame({"column": examples.keys(), "example_value": examples.values()}).reset_index(drop=True)

In [ ]:
df_rel = df_relationship[[
    "distance_miles",
    "recipient_matches_sender_customer_targets",
    "recipient_matches_sender_partner_targets",
    "recipient_cares_about_locality",
    "recipient_prefers_referral_partners",
    "same_state",
    "role_overlap",
    "sender_prefers_referral_partners",
    "mutual_count",
    "sender_cares_about_locality",
    "sender_matches_recipient_customer_targets",
    "sender_matches_recipient_partner_targets",
]].copy()

df_rel["distance"] = pd.cut(
    df_rel["distance_miles"].astype(float),
    bins=[-1, 30, 150, float("inf")],
    labels=["local (<30mi)", "regional (30-150mi)", "distant (150+mi)"],
)
df_rel = df_rel.drop(columns="distance_miles")

df_rel["strategy"] = df_context_analysis["STRATEGY"].values
df_rel["prompt_variant"] = df_context_analysis["DATA"].apply(
    lambda d: json.loads(d).get("context", {}).get("state", {}).get("prompt_variant")
).values

def _extract_business_model(data_str, party):
    """Extract sender or recipient business_model from the DATA JSON.
    Returns one of: 'b2b', 'b2c', 'both', 'none'."""
    try:
        d = json.loads(data_str)
        profile = d["context"]["state"]["prompt_context"]["input"][party]["profile_data"]
        section = profile.get(f"{party}_customer_section", {})
        val = section.get(f"{party}_characteristics", {}).get(f"{party}_business_model")
        if isinstance(val, list):
            items = {str(v).upper() for v in val}
        elif isinstance(val, str):
            items = {val.upper()}
        else:
            return "none"
        has_b2b = "B2B" in items
        has_b2c = "B2C" in items
        if has_b2b and has_b2c:
            return "both"
        elif has_b2b:
            return "b2b"
        elif has_b2c:
            return "b2c"
        else:
            return "none"
    except (KeyError, TypeError):
        return "none"

df_rel["sender_business_model"] = df_context_analysis["DATA"].apply(
    lambda d: _extract_business_model(d, "sender")
).values
df_rel["recipient_business_model"] = df_context_analysis["DATA"].apply(
    lambda d: _extract_business_model(d, "recipient")
).values

print(f"Sender business models: {df_rel['sender_business_model'].nunique()} unique, {df_rel['sender_business_model'].notna().sum():,} non-null")
print(df_rel["sender_business_model"].value_counts().head(10).to_string())
print(f"\nRecipient business models: {df_rel['recipient_business_model'].nunique()} unique, {df_rel['recipient_business_model'].notna().sum():,} non-null")
print(df_rel["recipient_business_model"].value_counts().head(10).to_string())

df_rel

In [ ]:
for col in ["sender_business_model", "recipient_business_model"]:
    print(df_rel[col].unique())

In [ ]:
# Extract tracking_id from DATA JSON for joining
df_context_analysis["tracking_id"] = df_context_analysis["DATA"].apply(
    lambda d: json.loads(d).get("tracking_id")
)

# Query: for each edit event, find the conversation and its most recent lifecycle state before the message was sent
df_lifecycle = query("""
    WITH edit_events AS (
        SELECT
            g.DATA:tracking_id::STRING as tracking_id,
            g.DATA:source_user_id::INTEGER as source_user_id,
            g.DATA:target_user_id::INTEGER as target_user_id,
            g.DATA:event_tstamp::TIMESTAMP_TZ as event_tstamp
        FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
        JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
            ON g.DATA:tracking_id = s.DATA:tracking_id
            AND s.DATA:event_name = 'llm_message_sent'
        WHERE g.DATA:event_name = 'llm_message_generated'
            AND s.DATA:edits_made::BOOLEAN = TRUE
    ),
    edit_with_conv AS (
        SELECT
            e.tracking_id,
            e.event_tstamp,
            c."id" as conversation_id
        FROM edit_events e
        JOIN OPENFLOW_DB."public"."conversations" c
            ON c."conversation_type" = 0
            AND (
                c."user_ids" = '{' || e.source_user_id || ',' || e.target_user_id || '}'
                OR c."user_ids" = '{' || e.target_user_id || ',' || e.source_user_id || '}'
            )
    ),
    lifecycle_transitions AS (
        SELECT
            properties:conversation_id::STRING as conversation_id,
            properties:new_lifecycle::STRING as lifecycle,
            event_tstamp
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_NONUSER_EVENTS
        WHERE event_name = 'meeting_lifecycle_transition'
    ),
    ranked AS (
        SELECT
            ec.tracking_id,
            lt.lifecycle,
            ROW_NUMBER() OVER (
                PARTITION BY ec.tracking_id
                ORDER BY lt.event_tstamp DESC
            ) as rn
        FROM edit_with_conv ec
        LEFT JOIN lifecycle_transitions lt
            ON lt.conversation_id = ec.conversation_id::STRING
            AND lt.event_tstamp <= ec.event_tstamp
    )
    SELECT
        tracking_id,
        COALESCE(lifecycle, 'none') as meeting_lifecycle
    FROM ranked
    WHERE rn = 1
""")

# Merge lifecycle into df_rel
lifecycle_map = df_lifecycle.set_index("TRACKING_ID")["MEETING_LIFECYCLE"]
df_rel["meeting_lifecycle"] = df_context_analysis["tracking_id"].map(lifecycle_map).fillna("none").values

print(f"Lifecycle distribution:")
print(df_rel["meeting_lifecycle"].value_counts().to_string())
df_rel

# Outcome Analysis

Let's take a look at how various aspects of edits affect outcomes: 
- A response was sent within 7 days
  	- Can look at the conversation_users table to see if the other conversation user in the conversation has activity after the edited message was sent
- A response was sent, within any amount of time
  	- Ditto: can use conversation_users
- Value moment in the conversation
  	- if last_value_moment_at is non-nil on the conversations table OR conversation_users table (both have this column and they mean the same thing for the same conversation) then both users sent >= 2 messages since the later of (conversation creation, last value moment, or the cutoff date). Can use the presence or lack of a value in this column as an outcome indicator.
- Conversation meeting lifecycle stage:
    - Values we care about as an outcome are one of
		- completed: they completed a 1:1 meeting with the recipient | the best outcome
		- scheduled: a meeting was scheduled and potentially happened | the next best outcome
		- mutual_intent: both parties agreed that they should meet, even if logistics weren't finalized | the next best outcome
	- can be queried from conversation_meeting_analyses on the meeting_lifecycle column


In [ ]:
# Pull outcome data for each edit event: recipient response, value moment, meeting lifecycle, contact info
# (connection request data joined separately below for performance)
df_outcomes = query("""
  WITH edit_events AS (
      SELECT
          g.DATA:tracking_id::STRING as tracking_id,
          g.DATA:source_user_id::INTEGER as source_user_id,
          g.DATA:target_user_id::INTEGER as target_user_id,
          g.DATA:event_tstamp::TIMESTAMP_TZ as event_tstamp,
          g.DATA:interaction_location::STRING as interaction_location
      FROM EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS g
      JOIN EVENTS.LLM_MESSAGES.LLM_MESSAGE_EVENTS s
          ON g.DATA:tracking_id = s.DATA:tracking_id
          AND s.DATA:event_name = 'llm_message_sent'
      WHERE g.DATA:event_name = 'llm_message_generated'
  ),
  edit_with_conv AS (
      SELECT
          e.tracking_id,
          e.source_user_id,
          e.target_user_id,
          e.event_tstamp,
          e.interaction_location,
          c."id" as conversation_id,
          c."last_value_moment_at" as conv_last_value_moment_at,
          ROW_NUMBER() OVER (PARTITION BY e.tracking_id ORDER BY c."last_message_at" DESC NULLS LAST) as rn
      FROM edit_events e
      JOIN OPENFLOW_DB."public"."conversations" c
          ON c."conversation_type" = 0
          AND (
              c."user_ids" = '{' || e.source_user_id || ',' || e.target_user_id || '}'
              OR c."user_ids" = '{' || e.target_user_id || ',' || e.source_user_id || '}'
          )
  ),
  base AS (
      SELECT * FROM edit_with_conv WHERE rn = 1
  )
  SELECT
      b.tracking_id,
      b.conversation_id,
      b.source_user_id,
      b.target_user_id,
      b.event_tstamp,
      b.interaction_location,
      IFF(b.conv_last_value_moment_at IS NOT NULL, TRUE, FALSE) as has_value_moment,
      COALESCE(
          CASE cma."meeting_lifecycle"
              WHEN 6 THEN 'completed'
              WHEN 5 THEN 'scheduled'
              WHEN 4 THEN 'mutual_intent'
              WHEN 3 THEN 'not_interested'
              WHEN 2 THEN 'proposed_didnt_work'
              WHEN 1 THEN 'proposed'
              WHEN 0 THEN 'none'
          END,
          'none'
      ) as meeting_lifecycle_outcome,
      IFF(cml_contact."conversation_id" IS NOT NULL, TRUE, FALSE) as has_contact_info
  FROM base b
  LEFT JOIN OPENFLOW_DB."public"."conversation_meeting_analyses" cma
      ON cma."conversation_id" = b.conversation_id
  LEFT JOIN (
      SELECT DISTINCT "conversation_id"
      FROM OPENFLOW_DB."public"."conversation_meeting_user_logistics"
      WHERE "field_type" IN (0, 1, 2, 5)
  ) cml_contact
      ON cml_contact."conversation_id" = b.conversation_id
""")

print(f"Snowflake outcomes loaded: {len(df_outcomes):,} rows")

# --- Connection request data: using events instead of business_requests table ---
import numpy as _np
from collections import defaultdict
from bisect import bisect_left, bisect_right

# Step 1: Query connection_request_sent events for our source users
source_user_ids = df_outcomes["SOURCE_USER_ID"].dropna().astype(int).unique().tolist()
print(f"Querying connection_request_sent events for {len(source_user_ids):,} source users...")

BATCH = 5000
sent_rows = []
for start in range(0, len(source_user_ids), BATCH):
    batch = source_user_ids[start:start + BATCH]
    ids_str = ",".join(str(u) for u in batch)
    sent_rows.extend(query(f"""
        SELECT
            USER_ID as source_user_id,
            PROPERTIES:target_user_id::INTEGER as target_user_id,
            BUSINESS_ID as source_business_id,
            PROPERTIES:target_business_id::INTEGER as target_business_id,
            EVENT_TSTAMP as sent_at
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_USER_EVENTS
        WHERE event_name = 'connection_request_sent'
            AND USER_ID IN ({ids_str})
            AND EVENT_TSTAMP >= '2026-01-01'::TIMESTAMP_TZ
    """).values.tolist())

print(f"Got {len(sent_rows):,} connection_request_sent events")

# Build lookup: (source_user_id, target_user_id) -> True
cr_sent_lookup = set()
for src_uid, tgt_uid, src_biz, tgt_biz, sent_at in sent_rows:
    if pd.notna(src_uid) and pd.notna(tgt_uid):
        cr_sent_lookup.add((int(src_uid), int(tgt_uid)))

df_outcomes["HAS_CONNECTION_REQUEST"] = [
    (int(s), int(t)) in cr_sent_lookup if pd.notna(s) and pd.notna(t) else False
    for s, t in zip(df_outcomes["SOURCE_USER_ID"], df_outcomes["TARGET_USER_ID"])
]
df_outcomes["IS_CR_SENDING_LOCATION"] = df_outcomes["INTERACTION_LOCATION"].isin([
    "connection_request_introduce_yourself_llm_modal",
    "daily_action_plan_introduce_yourself_llm",
])

print(f"HAS_CONNECTION_REQUEST: {sum(df_outcomes['HAS_CONNECTION_REQUEST']):,} / {len(df_outcomes):,}")

# Step 2: Query connection_request_accepted events for our target users
# Use business_id matching since accepted events have source_business_id
# First build user_to_biz from the sent events (no need for separate users table query)
user_to_biz = {}
for src_uid, tgt_uid, src_biz, tgt_biz, sent_at in sent_rows:
    if pd.notna(src_uid) and pd.notna(src_biz):
        user_to_biz[int(src_uid)] = int(src_biz)
    if pd.notna(tgt_uid) and pd.notna(tgt_biz):
        user_to_biz[int(tgt_uid)] = int(tgt_biz)

print(f"Built user_to_biz mapping for {len(user_to_biz):,} users (from sent events)")

all_biz_ids = list(set(user_to_biz.values()))
print(f"Querying connection_request_accepted events for {len(all_biz_ids):,} businesses...")

acceptance_rows = []
for start in range(0, len(all_biz_ids), BATCH):
    batch = all_biz_ids[start:start + BATCH]
    ids_str = ",".join(str(b) for b in batch)
    acceptance_rows.extend(query(f"""
        SELECT
            BUSINESS_ID as acceptor_business_id,
            PROPERTIES:source_business_id::INTEGER as source_business_id,
            EVENT_TSTAMP as accepted_at
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_USER_EVENTS
        WHERE event_name = 'connection_request_accepted'
            AND BUSINESS_ID IN ({ids_str})
            AND EVENT_TSTAMP >= '2026-01-01'::TIMESTAMP_TZ
    """).values.tolist())

print(f"Got {len(acceptance_rows):,} acceptance events")

# Build lookup: (source_biz, acceptor_biz) -> sorted list of acceptance timestamps
acceptance_by_pair = defaultdict(list)
for acceptor_biz, source_biz, tstamp in acceptance_rows:
    if pd.notna(acceptor_biz) and pd.notna(source_biz) and pd.notna(tstamp):
        ts = pd.Timestamp(tstamp)
        if ts.tzinfo is None:
            ts = ts.tz_localize("UTC")
        acceptance_by_pair[(int(source_biz), int(acceptor_biz))].append(ts)

for key in acceptance_by_pair:
    acceptance_by_pair[key].sort()

# Step 3: For each CR-sending edit, check if accepted within 7 days
def _accepted_within_7d(source_user_id, target_user_id, event_tstamp):
    if pd.isna(source_user_id) or pd.isna(target_user_id) or pd.isna(event_tstamp):
        return False
    s_biz = user_to_biz.get(int(source_user_id))
    t_biz = user_to_biz.get(int(target_user_id))
    if s_biz is None or t_biz is None:
        return False
    acceptances = acceptance_by_pair.get((s_biz, t_biz), [])
    if not acceptances:
        return False
    et = pd.Timestamp(event_tstamp)
    if et.tzinfo is None:
        et = et.tz_localize("UTC")
    lo = bisect_left(acceptances, et)
    hi = bisect_right(acceptances, et + pd.Timedelta(days=7))
    return lo < hi

_accepted_7d = [
    _accepted_within_7d(s, t, ts)
    for s, t, ts in zip(df_outcomes["SOURCE_USER_ID"], df_outcomes["TARGET_USER_ID"], df_outcomes["EVENT_TSTAMP"])
]

# Only meaningful for CR-sending locations; NaN otherwise
df_outcomes["CONNECTION_ACCEPTED_7D"] = _np.where(
    df_outcomes["IS_CR_SENDING_LOCATION"],
    _accepted_7d,
    _np.nan
)

_cr_sending = df_outcomes[df_outcomes["IS_CR_SENDING_LOCATION"]]
print(f"\nConnection accepted within 7d (CR-sending only): {_cr_sending['CONNECTION_ACCEPTED_7D'].sum():.0f} / {len(_cr_sending):,} ({_cr_sending['CONNECTION_ACCEPTED_7D'].mean():.1%})")
print(f"CR-sending locations: {df_outcomes['IS_CR_SENDING_LOCATION'].sum():,}")


In [ ]:
# --- Query conversation_message events to determine if recipient responded within 7 days ---
# Look for events where the target user sent a message back to the source user
target_user_ids = df_outcomes["TARGET_USER_ID"].dropna().astype(int).unique().tolist()

print(f"Querying conversation_message events for {len(target_user_ids):,} target users...")

BATCH_SIZE = 5000
reply_rows = []
for start in range(0, len(target_user_ids), BATCH_SIZE):
    batch = target_user_ids[start:start + BATCH_SIZE]
    ids_str = ",".join(str(u) for u in batch)
    reply_rows.extend(query(f"""
        SELECT
            USER_ID,
            PROPERTIES:target_user_id::INTEGER as target_user_id,
            EVENT_TSTAMP
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_USER_EVENTS
        WHERE event_name = 'conversation_message'
            AND USER_ID IN ({ids_str})
            AND EVENT_TSTAMP >= '2026-01-01'::TIMESTAMP_TZ
    """).values.tolist())

print(f"Got {len(reply_rows):,} reply message events")

# Build lookup: (sender_user_id, recipient_user_id) -> sorted list of message timestamps
from collections import defaultdict
from bisect import bisect_left, bisect_right

replies_by_pair = defaultdict(list)
for sender, recipient, tstamp in reply_rows:
    if pd.notna(sender) and pd.notna(recipient) and pd.notna(tstamp):
        ts = pd.Timestamp(tstamp)
        if ts.tzinfo is None:
            ts = ts.tz_localize("UTC")
        replies_by_pair[(int(sender), int(recipient))].append(ts)

for key in replies_by_pair:
    replies_by_pair[key].sort()

print(f"Built reply lookup for {len(replies_by_pair):,} unique (sender, recipient) pairs")

# For each edit, check if the target sent a message to the source within 7 days
def _recipient_responded_7d(source_user_id, target_user_id, event_tstamp):
    if pd.isna(source_user_id) or pd.isna(target_user_id) or pd.isna(event_tstamp):
        return False
    # Target replying to source = target sent a message to source
    replies = replies_by_pair.get((int(target_user_id), int(source_user_id)), [])
    if not replies:
        return False
    et = pd.Timestamp(event_tstamp)
    if et.tzinfo is None:
        et = et.tz_localize("UTC")
    lo = bisect_left(replies, et)
    hi = bisect_right(replies, et + pd.Timedelta(days=7))
    return lo < hi

df_outcomes["RECIPIENT_RESPONDED_7D"] = [
    _recipient_responded_7d(s, t, ts)
    for s, t, ts in zip(df_outcomes["SOURCE_USER_ID"], df_outcomes["TARGET_USER_ID"], df_outcomes["EVENT_TSTAMP"])
]

# Merge outcomes into df_rel via tracking_id (deduplicate in case business_requests join produced dupes)
outcomes_indexed = df_outcomes.drop_duplicates(subset="TRACKING_ID", keep="first").set_index("TRACKING_ID")
df_rel["recipient_responded"] = df_context_analysis["tracking_id"].map(outcomes_indexed["RECIPIENT_RESPONDED_7D"]).fillna(False).values
df_rel["has_value_moment"] = df_context_analysis["tracking_id"].map(outcomes_indexed["HAS_VALUE_MOMENT"]).fillna(False).values
df_rel["meeting_lifecycle_outcome"] = df_context_analysis["tracking_id"].map(outcomes_indexed["MEETING_LIFECYCLE_OUTCOME"]).fillna("none").values
df_rel["has_contact_info"] = df_context_analysis["tracking_id"].map(outcomes_indexed["HAS_CONTACT_INFO"]).fillna(False).values
df_rel["has_connection_request"] = df_context_analysis["tracking_id"].map(outcomes_indexed["HAS_CONNECTION_REQUEST"]).fillna(False).values
df_rel["is_cr_sending_location"] = df_context_analysis["tracking_id"].map(outcomes_indexed["IS_CR_SENDING_LOCATION"]).fillna(False).values
# connection_accepted_7d already NaN for non-CR locations (set at source)
df_rel["connection_accepted"] = df_context_analysis["tracking_id"].map(outcomes_indexed["CONNECTION_ACCEPTED_7D"]).values

print(f"\nRecipient responded within 7d:")
print(df_rel["recipient_responded"].value_counts().to_string())
print(f"\nHas value moment:")
print(df_rel["has_value_moment"].value_counts().to_string())
print(f"\nMeeting lifecycle outcome:")
print(df_rel["meeting_lifecycle_outcome"].value_counts().to_string())
print(f"\nHas contact info shared:")
print(df_rel["has_contact_info"].value_counts().to_string())
print(f"\nConnection request associated:")
print(df_rel["has_connection_request"].value_counts().to_string())
print(f"\nConnection accepted within 7d (CR-sending only):")
cr_sending = df_rel[df_rel["is_cr_sending_location"] == True]
_ca = pd.to_numeric(cr_sending["connection_accepted"], errors="coerce")
print(f"  {_ca.sum():.0f} / {len(cr_sending):,} ({_ca.mean():.1%})")

In [ ]:
# Diagnostic: trace CONNECTION_ACCEPTED_7D
print("1. df_outcomes CONNECTION_ACCEPTED_7D value counts:")
print(df_outcomes["CONNECTION_ACCEPTED_7D"].value_counts(dropna=False).to_string())

print(f"\n2. acceptance_by_pair size: {len(acceptance_by_pair):,}")
print(f"   Total acceptance events: {sum(len(v) for v in acceptance_by_pair.values()):,}")

# Test with a specific CR-sending row
cr_rows = df_outcomes[df_outcomes["IS_CR_SENDING_LOCATION"] == True]
if len(cr_rows) > 0:
    row = cr_rows.iloc[0]
    s, t = row["SOURCE_USER_ID"], row["TARGET_USER_ID"]
    s_biz = user_to_biz.get(int(s))
    t_biz = user_to_biz.get(int(t))
    print(f"\n3. First CR-sending row:")
    print(f"   source_user={int(s)}, target_user={int(t)}")
    print(f"   source_biz={s_biz}, target_biz={t_biz}")
    print(f"   event_tstamp={row['EVENT_TSTAMP']}")
    
    key = (s_biz, t_biz) if s_biz and t_biz else None
    print(f"   acceptance_by_pair key {key}: {acceptance_by_pair.get(key, 'NOT FOUND')}")
    key_rev = (t_biz, s_biz) if s_biz and t_biz else None
    print(f"   acceptance_by_pair key reversed {key_rev}: {acceptance_by_pair.get(key_rev, 'NOT FOUND')}")
    
    # Check what keys exist for these businesses
    matching_keys = [k for k in acceptance_by_pair if s_biz in k or (t_biz is not None and t_biz in k)]
    print(f"   Keys involving either biz: {matching_keys[:5]}")

# Sample a row that has HAS_CONNECTION_REQUEST=True
cr_with_request = cr_rows[cr_rows["HAS_CONNECTION_REQUEST"] == True]
print(f"\n4. CR-sending rows with HAS_CONNECTION_REQUEST=True: {len(cr_with_request):,} / {len(cr_rows):,}")

if len(cr_with_request) > 0:
    row = cr_with_request.iloc[0]
    s_biz = user_to_biz.get(int(row["SOURCE_USER_ID"]))
    t_biz = user_to_biz.get(int(row["TARGET_USER_ID"]))
    print(f"\n5. CR-sending row WITH connection request:")
    print(f"   source_biz={s_biz}, target_biz={t_biz}, event_tstamp={row['EVENT_TSTAMP']}")
    for key in [(s_biz, t_biz), (t_biz, s_biz)]:
        print(f"   acceptance_by_pair[{key}]: {acceptance_by_pair.get(key, 'NOT FOUND')}")

In [ ]:
# Sanity check: overall 7-day connection acceptance rate (last 90 days, non-Explorer only)
df_cr_baseline = query("""
    WITH sent AS (
        SELECT
            e.business_id as sender_business_id,
            e.event_tstamp as sent_at
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_USER_EVENTS e
        JOIN ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES b ON b.id = e.business_id
        WHERE e.event_name = 'connection_request_sent'
            AND b.plan_tier_name != 'Explorer'
            AND e.event_tstamp >= DATEADD(DAY, -90, CURRENT_TIMESTAMP())
    ),
    accepted AS (
        SELECT
            e.PROPERTIES:source_business_id::INTEGER as sender_business_id,
            e.business_id as acceptor_business_id,
            e.event_tstamp as accepted_at
        FROM ALIGNABLE_REPORTING.EVENT_STREAMS.REAL_TIME_USER_EVENTS e
        JOIN ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES b ON b.id = e.business_id
        WHERE e.event_name = 'connection_request_accepted'
            AND b.plan_tier_name != 'Explorer'
            AND e.event_tstamp >= DATEADD(DAY, -90, CURRENT_TIMESTAMP())
    ),
    matched AS (
        SELECT
            s.sender_business_id,
            s.sent_at,
            MAX(CASE WHEN a.accepted_at IS NOT NULL THEN 1 ELSE 0 END) as accepted_within_7d
        FROM sent s
        LEFT JOIN accepted a
            ON a.sender_business_id = s.sender_business_id
            AND a.accepted_at >= s.sent_at
            AND a.accepted_at <= DATEADD(DAY, 7, s.sent_at)
        GROUP BY s.sender_business_id, s.sent_at
    )
    SELECT
        COUNT(*) as total_sent,
        SUM(accepted_within_7d) as accepted_within_7d,
        ROUND(SUM(accepted_within_7d) / NULLIF(COUNT(*), 0) * 100, 2) as acceptance_rate_7d_pct
    FROM matched
""")

print("Baseline connection acceptance rate within 7 days (last 90 days, non-Explorer):")
print(f"  Sent: {df_cr_baseline['TOTAL_SENT'].iloc[0]:,}")
print(f"  Accepted within 7d: {df_cr_baseline['ACCEPTED_WITHIN_7D'].iloc[0]:,}")
print(f"  7d acceptance rate: {df_cr_baseline['ACCEPTANCE_RATE_7D_PCT'].iloc[0]}%")

# Compare to our edit data
_cr_sending = df_rel[df_rel["is_cr_sending_location"] == True]
_ca = pd.to_numeric(_cr_sending["connection_accepted"], errors="coerce")
print(f"\nOur edit data (CR-sending locations, 7d window):")
print(f"  Accepted: {_ca.sum():.0f} / {len(_cr_sending):,} ({_ca.mean():.1%})")

In [ ]:
# Build combined dataframe: edit data + context_ prefixed columns + outcome_ prefixed columns

# Context columns from df_rel (relationship data, strategy, prompt variant, meeting lifecycle at send time)
context_cols = {
    "distance": "context_distance",
    "recipient_matches_sender_customer_targets": "context_recipient_matches_sender_customer_targets",
    "recipient_matches_sender_partner_targets": "context_recipient_matches_sender_partner_targets",
    "recipient_cares_about_locality": "context_recipient_cares_about_locality",
    "recipient_prefers_referral_partners": "context_recipient_prefers_referral_partners",
    "same_state": "context_same_state",
    "role_overlap": "context_role_overlap",
    "sender_prefers_referral_partners": "context_sender_prefers_referral_partners",
    "mutual_count": "context_mutual_count",
    "sender_cares_about_locality": "context_sender_cares_about_locality",
    "sender_matches_recipient_customer_targets": "context_sender_matches_recipient_customer_targets",
    "sender_matches_recipient_partner_targets": "context_sender_matches_recipient_partner_targets",
    "strategy": "context_strategy",
    "prompt_variant": "context_prompt_variant",
    "meeting_lifecycle": "context_meeting_lifecycle",
    "sender_business_model": "context_sender_business_model",
    "recipient_business_model": "context_recipient_business_model",
}

# Outcome columns from df_rel
outcome_cols = {
    "recipient_responded": "outcome_recipient_responded",
    "has_value_moment": "outcome_has_value_moment",
    "meeting_lifecycle_outcome": "outcome_meeting_lifecycle",
    "has_contact_info": "outcome_has_contact_info",
    "has_connection_request": "outcome_has_connection_request",
    "connection_accepted": "outcome_connection_accepted",  # now uses 7-day window from events
}

# Start with edit data from df_context_analysis
df_combined = df_context_analysis[["SOURCE_USER_ID", "TARGET_USER_ID", "GENERATED_MESSAGE", "SENT_MESSAGE", "STRATEGY", "INTERACTION_LOCATION"]].copy()
df_combined.columns = df_combined.columns.str.lower()

# Strip surrounding quotes from Snowflake string columns
df_combined["interaction_location"] = df_combined["interaction_location"].str.strip('"')

# Add context columns
for old, new in context_cols.items():
    if old in df_rel.columns:
        df_combined[new] = df_rel[old].values

# Add outcome columns
for old, new in outcome_cols.items():
    if old in df_rel.columns:
        df_combined[new] = df_rel[old].values

print(f"Combined dataframe: {len(df_combined):,} rows, {len(df_combined.columns)} columns")
print(f"\nColumns:")
for col in df_combined.columns:
    print(f"  {col}")

df_combined.head()

In [ ]:
# Enrich df_combined with edit annotation data (theme, actionability, etc.) from the clustering pipeline
# Normalize messages to match df_edits preprocessing (em-dash normalization + strip)
df_combined["_gen_norm"] = df_combined["generated_message"].str.strip().str.replace("—", ", ", regex=False).str.replace("–", ", ", regex=False)
df_combined["_sent_norm"] = df_combined["sent_message"].str.strip().str.replace("—", ", ", regex=False).str.replace("–", ", ", regex=False)

_edit_data = df_edits[["GENERATED_MESSAGE", "SENT_MESSAGE", "theme", "actionability_delta", "message_quality_impact", "edit_actions"]].drop_duplicates(
    subset=["GENERATED_MESSAGE", "SENT_MESSAGE"], keep="first"
)

df_combined = df_combined.merge(
    _edit_data,
    left_on=["_gen_norm", "_sent_norm"],
    right_on=["GENERATED_MESSAGE", "SENT_MESSAGE"],
    how="left",
).drop(columns=["_gen_norm", "_sent_norm", "GENERATED_MESSAGE", "SENT_MESSAGE"])

# Positive meeting = scheduled or completed
df_combined["outcome_positive_meeting"] = df_combined["outcome_meeting_lifecycle"].isin(["scheduled", "completed"])

# Maybe met externally = mutual_intent with contact info shared (plausibly connected outside the platform)
df_combined["outcome_maybe_met_externally"] = (
    (df_combined["outcome_meeting_lifecycle"] == "mutual_intent")
    & (df_combined["outcome_has_contact_info"] == True)
)

matched = df_combined["theme"].notna().sum()
print(f"Matched {matched:,}/{len(df_combined):,} rows with edit annotation data ({matched/len(df_combined):.1%})")
print(f"\nPositive meeting (scheduled/completed): {df_combined['outcome_positive_meeting'].sum():,} ({df_combined['outcome_positive_meeting'].mean():.1%})")
print(f"Maybe met externally (mutual_intent + contact info): {df_combined['outcome_maybe_met_externally'].sum():,} ({df_combined['outcome_maybe_met_externally'].mean():.1%})")
print(f"\nFinal df_combined: {len(df_combined):,} rows, {len(df_combined.columns)} columns")
df_combined.head()

In [ ]:
# Save / load df_combined to survive kernel crashes
import os, pickle

df_combined_path = "/Users/garrettsmith/notebooks/edit analysis/.llm_cache/df_combined.pkl"

if os.path.exists(df_combined_path) and 'df_combined' not in dir():
    df_combined = pickle.load(open(df_combined_path, "rb"))
    print(f"Loaded df_combined from cache: {len(df_combined):,} rows, {len(df_combined.columns)} columns")
else:
    pickle.dump(df_combined, open(df_combined_path, "wb"))
    print(f"Saved df_combined to {df_combined_path}: {len(df_combined):,} rows, {len(df_combined.columns)} columns")

In [ ]:
# Filter out rows where sent_message is empty — these are generated-only records
# that were never actually sent and skew outcome stats.
_before = len(df_combined)
df_combined = df_combined[df_combined["sent_message"].str.strip().astype(bool)].copy()
print(f"Filtered empty sent_message: {_before:,} → {len(df_combined):,} ({_before - len(df_combined):,} removed)")

In [ ]:
df_combined.head(1)

In [ ]:
len(df_combined[df_combined["outcome_connection_accepted"] == True]) / len(df_combined)

In [ ]:
# Helper: grouped bar chart showing outcome rates by a categorical column
def plot_outcome_rates(df, group_col, title, min_n=20, figsize=(14, 6), preserve_order=False, sort_by=None, include_connection_accepted=False):
    """For each group, show outcome rates.
    Set include_connection_accepted=True to add Connection Accepted bar.
    Returns (fig, grouped_df)."""
    # Ensure connection_accepted is numeric (may contain NaN for non-CR locations)
    if "outcome_connection_accepted" in df.columns:
        df = df.copy()
        df["outcome_connection_accepted"] = pd.to_numeric(df["outcome_connection_accepted"], errors="coerce")

    agg_dict = {
        "n": ("outcome_recipient_responded", "size"),
        "recipient_responded": ("outcome_recipient_responded", "mean"),
        "value_moment": ("outcome_has_value_moment", "mean"),
        "positive_meeting": ("outcome_positive_meeting", "mean"),
        "maybe_met": ("outcome_maybe_met_externally", "mean"),
    }
    if include_connection_accepted:
        agg_dict["connection_accepted"] = ("outcome_connection_accepted", "mean")

    grouped = df.groupby(group_col).agg(**agg_dict)
    grouped = grouped[grouped["n"] >= min_n]
    if sort_by:
        grouped = grouped.sort_values(sort_by, ascending=False)
    elif not preserve_order:
        grouped = grouped.sort_values("n", ascending=False)

    cols = ["recipient_responded", "value_moment", "maybe_met", "positive_meeting"]
    labels = ["Recipient Responded", "Value Moment", "Likely Met", "Met 1:1"]
    colors = ["goldenrod", "coral", "seagreen", "mediumpurple"]
    if include_connection_accepted:
        cols.insert(0, "connection_accepted")
        labels.insert(0, "Connection Accepted")
        colors.insert(0, "steelblue")

    x = range(len(grouped))
    n_bars = len(cols)
    width = 0.18 if n_bars <= 4 else 0.16
    fig, ax = plt.subplots(figsize=figsize)

    offsets = [(i - (n_bars - 1) / 2) * width for i in range(n_bars)]
    for offset, col, label, color in zip(offsets, cols, labels, colors):
        positions = [i + offset for i in x]
        values = grouped[col] * 100
        bars = ax.bar(positions, values, width, label=label, color=color)
        for bar, val in zip(bars, values):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                        f"{val:.1f}%", ha="center", va="bottom", fontsize=5, color="black")

    ax.set_xticks(list(x))
    ax.set_xticklabels(grouped.index, rotation=45, ha="right", fontsize=8)

    y_top = ax.get_ylim()[1]
    for i, n in enumerate(grouped["n"]):
        ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")

    ax.set_ylabel("% of edits")
    ax.set_title(title, pad=25)
    ax.legend(loc="upper right", fontsize=7)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    return fig, grouped

### 1. Edit Theme x Outcome Rates
For each edit theme, what % of edits led to follow-up activity, a value moment, or a positive meeting outcome?

In [ ]:
df_with_theme = df_combined[df_combined["theme"].notna()]
fig, theme_outcomes = plot_outcome_rates(df_with_theme, "theme", "Outcome Rates by Edit Theme", sort_by="recipient_responded")
plt.show()

### 2. Actionability Delta x Outcome Rates
Do users who make messages less actionable actually get worse outcomes?

In [ ]:
df_with_action = df_combined[df_combined["actionability_delta"].notna()]
action_order = ["much_more_actionable", "slightly_more_actionable", "neutral", "slightly_less_actionable", "much_less_actionable"]
df_with_action["actionability_delta"] = pd.Categorical(df_with_action["actionability_delta"], categories=action_order, ordered=True)
fig, actionability_outcomes = plot_outcome_rates(df_with_action, "actionability_delta", "Outcome Rates by Actionability Delta")
plt.show()

### 3. Strategy x Outcome Rates
Which message strategies drive the best raw outcomes?

In [ ]:
fig, strategy_outcomes = plot_outcome_rates(df_combined, "context_strategy", "Outcome Rates by Strategy")
plt.show()

### 3b. Prompt Variant x Edit Theme Heatmap
Which prompt variants produce which edit patterns?

In [ ]:
def plot_variant_x_theme():
    df_pv = df_combined[df_combined["theme"].notna() & df_combined["context_prompt_variant"].notna()]
    ct = pd.crosstab(df_pv["context_prompt_variant"], df_pv["theme"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    theme_order = df_pv["theme"].value_counts().index.tolist()
    ct_pct = ct_pct[[c for c in theme_order if c in ct_pct.columns]]

    fig, ax = plt.subplots(figsize=(max(16, len(ct_pct.columns) * 1.1), max(4, len(ct_pct) * 0.8)))
    im = ax.imshow(ct_pct.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="% of edits in variant")
    ax.set_xticks(range(len(ct_pct.columns))); ax.set_xticklabels(ct_pct.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(ct_pct.index))); ax.set_yticklabels(ct_pct.index, fontsize=9)
    ax.set_title("Edit Theme Distribution by Prompt Variant (row-normalised %)")
    for i in range(len(ct_pct.index)):
        for j in range(len(ct_pct.columns)):
            val = ct_pct.values[i, j]
            if val > 0.5:
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6,
                        color="black" if val < ct_pct.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig

fig = plot_variant_x_theme()
plt.show()

### 4. Prompt Variant x Outcome Rates
Which prompt variants drive the best outcomes?

In [ ]:
fig, variant_outcomes = plot_outcome_rates(df_combined, "context_prompt_variant", "Outcome Rates by Prompt Variant", sort_by="recipient_responded")
plt.show()

### 5. Strategy x Edit Theme Heatmap
Which strategies produce which edit patterns?

In [ ]:
def plot_strategy_x_theme():
    df_st = df_combined[df_combined["theme"].notna()]
    ct = pd.crosstab(df_st["context_strategy"], df_st["theme"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    theme_order = df_st["theme"].value_counts().index.tolist()
    ct_pct = ct_pct[[c for c in theme_order if c in ct_pct.columns]]

    fig, ax = plt.subplots(figsize=(max(16, len(ct_pct.columns) * 1.1), max(4, len(ct_pct) * 0.8)))
    im = ax.imshow(ct_pct.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="% of edits in strategy")
    ax.set_xticks(range(len(ct_pct.columns))); ax.set_xticklabels(ct_pct.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(ct_pct.index))); ax.set_yticklabels(ct_pct.index, fontsize=9)
    ax.set_title("Edit Theme Distribution by Strategy (row-normalised %)")
    for i in range(len(ct_pct.index)):
        for j in range(len(ct_pct.columns)):
            val = ct_pct.values[i, j]
            if val > 0.5:
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=6,
                        color="black" if val < ct_pct.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig

fig = plot_strategy_x_theme()
plt.show()

### 6. Distance x Outcome Rates
Does proximity to the recipient predict better outcomes?

In [ ]:
df_with_dist = df_combined[df_combined["context_distance"].notna()]
fig, distance_outcomes = plot_outcome_rates(df_with_dist, "context_distance", "Outcome Rates by Distance to Recipient")
plt.show()

### 7. Distance x Edit Theme Heatmap
Do users edit messages differently depending on how far away the recipient is?

In [ ]:
def plot_distance_x_theme():
    df_dt = df_combined[df_combined["theme"].notna() & df_combined["context_distance"].notna()]
    ct = pd.crosstab(df_dt["context_distance"], df_dt["theme"])
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    theme_order = df_dt["theme"].value_counts().index.tolist()
    ct_pct = ct_pct[[c for c in theme_order if c in ct_pct.columns]]

    fig, ax = plt.subplots(figsize=(max(16, len(ct_pct.columns) * 1.1), max(4, len(ct_pct) * 1.2)))
    im = ax.imshow(ct_pct.values, aspect="auto", cmap="YlGnBu")
    plt.colorbar(im, ax=ax, label="% of edits in distance bucket")
    ax.set_xticks(range(len(ct_pct.columns))); ax.set_xticklabels(ct_pct.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(ct_pct.index))); ax.set_yticklabels(ct_pct.index, fontsize=9)
    ax.set_title("Edit Theme Distribution by Distance (row-normalised %)")
    for i in range(len(ct_pct.index)):
        for j in range(len(ct_pct.columns)):
            val = ct_pct.values[i, j]
            if val > 0.5:
                ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=7,
                        color="black" if val < ct_pct.values.max() * 0.6 else "white")
    fig.tight_layout()
    return fig

fig = plot_distance_x_theme()
plt.show()

### 8. Mutual Connections x Outcome Rates
Does having mutual connections predict better outcomes?

In [ ]:
df_combined["context_mutual_bucket"] = pd.cut(
    df_combined["context_mutual_count"].fillna(0).astype(float),
    bins=[-1, 0, 3, 10, float("inf")],
    labels=["0 mutuals", "1-3 mutuals", "4-10 mutuals", "11+ mutuals"],
)

fig, mutual_outcomes = plot_outcome_rates(df_combined, "context_mutual_bucket", "Outcome Rates by Mutual Connection Count")
plt.show()

### 9. Meeting Lifecycle at Send Time x Outcome
Does existing relationship momentum predict better outcomes?

In [ ]:
fig, lifecycle_outcomes = plot_outcome_rates(df_combined, "context_meeting_lifecycle", "Outcome Rates by Meeting Lifecycle at Send Time")
plt.show()

### 10. Degree of Editing x Outcome Rates
Does the percentage of the generated message that was changed affect outcomes?

In [ ]:
import difflib

def pct_changed(gen, sent):
    if not gen: return 1.0
    sm = difflib.SequenceMatcher(None, gen, sent)
    matching = sum(block.size for block in sm.get_matching_blocks())
    return 1.0 - (matching / len(gen))

df_combined["pct_changed"] = [
    pct_changed(g or "", s or "")
    for g, s in zip(df_combined["generated_message"], df_combined["sent_message"])
]
df_combined["pct_changed_bucket"] = pd.cut(
    df_combined["pct_changed"],
    bins=[-0.001, 0.001, 0.20, 0.40, 0.60, 0.80, 0.998, 1.001],
    labels=["0%", "1-20%", "21-40%", "41-60%", "61-80%", "81-99%", "100%"],
)

print("Distribution of edit degree:")
print(df_combined["pct_changed_bucket"].value_counts().sort_index().to_string())

fig, edit_degree_outcomes = plot_outcome_rates(
    df_combined, "pct_changed_bucket",
    "Outcome Rates by Degree of Editing (% of message changed)", preserve_order=True,
)
plt.show()

In [ ]:
pct_100_edits = df_combined[df_combined["pct_changed_bucket"] == "100%"]
pct_100_edits["outcome_recipient_responded"]

In [ ]:
# pct_100_edits.head(1)["sent_message"].item()
len(pct_100_edits[pct_100_edits["sent_message"] == ""])

### 11. Customer vs Partner Targeting x Outcome Rates
Do messages sent to customer-targeted recipients vs partner-targeted recipients have different outcomes?

In [ ]:
def classify_targeting(row):
    is_customer = row.get("context_recipient_matches_sender_customer_targets") == True
    is_partner = row.get("context_recipient_matches_sender_partner_targets") == True
    if is_customer and is_partner: return "Both"
    elif is_customer: return "Customer"
    elif is_partner: return "Partner"
    else: return "Neither"

df_combined["recipient_targeting"] = df_combined.apply(classify_targeting, axis=1)
print("Targeting distribution:")
print(df_combined["recipient_targeting"].value_counts().to_string())

fig, targeting_outcomes = plot_outcome_rates(df_combined[df_combined["pct_changed_bucket"] == "0%"], "recipient_targeting", "Outcome Rates by Recipient Targeting (Customer vs Partner) - Unedited Messages Only")
plt.show()

In [ ]:
df_combined["recipient_targeting"] = df_combined.apply(classify_targeting, axis=1)
print("Targeting distribution:")
print(df_combined["recipient_targeting"].value_counts().to_string())

fig, targeting_outcomes = plot_outcome_rates(df_combined[df_combined["pct_changed_bucket"] != "0%"], "recipient_targeting", "Outcome Rates by Recipient Targeting (Customer vs Partner) - Edited Messages Only")
plt.show()

### 12. Top Edit Actions x Outcome Rates
For each edit action appearing in >10% of edits, what are the outcome rates?

In [ ]:
def plot_top_action_outcomes():
    df_with_actions = df_combined[df_combined["edit_actions"].notna()].copy()
    total_edits = len(df_with_actions)

    action_counts = Counter()
    for actions in df_with_actions["edit_actions"]:
        if isinstance(actions, list):
            for a in actions: action_counts[a] += 1

    top_actions = [a for a, c in action_counts.items() if c / total_edits > 0.10]
    top_actions.sort(key=lambda a: -action_counts[a])

    action_outcome_rows = []
    for action in top_actions:
        mask = df_with_actions["edit_actions"].apply(lambda a: isinstance(a, list) and action in a)
        subset = df_with_actions[mask]
        action_outcome_rows.append({
            "edit_action": action, "n": len(subset),
            "recipient_responded": subset["outcome_recipient_responded"].mean() * 100,
            "connection_accepted": subset["outcome_connection_accepted"].mean() * 100,
            "value_moment": subset["outcome_has_value_moment"].mean() * 100,
            "positive_meeting": subset["outcome_positive_meeting"].mean() * 100,
            "maybe_met": subset["outcome_maybe_met_externally"].mean() * 100,
        })
    df_ao = pd.DataFrame(action_outcome_rows).sort_values("recipient_responded", ascending=False).reset_index(drop=True)

    x = range(len(df_ao)); width = 0.16
    fig, ax = plt.subplots(figsize=(14, 6))
    n_b = 5; offsets = [(i - (n_b-1)/2)*width for i in range(n_b)]
    cols = ["connection_accepted", "recipient_responded", "value_moment", "positive_meeting", "maybe_met"]
    labels = ["Connection Accepted", "Recipient Responded", "Value Moment", "Met 1:1", "Likely Met"]
    colors = ["steelblue", "goldenrod", "coral", "seagreen", "mediumpurple"]
    for offset, col, label, color in zip(offsets, cols, labels, colors):
        pos = [i + offset for i in x]; vals = df_ao[col]
        bars = ax.bar(pos, vals, width, label=label, color=color)
        for bar, val in zip(bars, vals):
            if val > 0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f"{val:.1f}%", ha="center", va="bottom", fontsize=5, color="black")
    ax.set_xticks(list(x)); ax.set_xticklabels(df_ao["edit_action"], rotation=45, ha="right", fontsize=8)
    y_top = ax.get_ylim()[1]
    for i, n in enumerate(df_ao["n"]): ax.text(i, y_top*1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")
    ax.set_ylabel("% of edits"); ax.set_title("Outcome Rates by Top Edit Action (>10% of edits)", pad=25)
    ax.legend(loc="upper right", fontsize=7); fig.tight_layout(); plt.subplots_adjust(top=0.90)
    return fig

fig = plot_top_action_outcomes()
plt.show()

### 13. Prompt Variant x Outcomes — Unedited Messages Only (0% change)
For messages users sent without any changes, how do outcomes vary by prompt variant?

In [ ]:
df_no_edit = df_combined[df_combined["pct_changed_bucket"] == "0%"]
print(f"Unedited messages (0% change): {len(df_no_edit):,}")

fig, variant_outcomes_no_edit = plot_outcome_rates(
    df_no_edit, "context_prompt_variant",
    "Outcome Rates by Prompt Variant — Unedited Messages (0% change)",
    sort_by="recipient_responded", min_n=5,
)
plt.show()

### 14. Interaction Location x Outcomes — Unedited Messages Only (0% change)
For messages users sent without any changes, how do outcomes vary by interaction location?

In [ ]:
def plot_location_outcomes_no_edit():
    df_no_edit_loc = df_no_edit.copy()
    df_no_edit_loc["interaction_location_label"] = df_no_edit_loc["interaction_location"].map(LOCATION_LABELS).fillna(df_no_edit_loc["interaction_location"])
    print("Interaction locations in unedited messages:")
    print(df_no_edit_loc["interaction_location_label"].value_counts().to_string())
    return plot_outcome_rates(
        df_no_edit_loc, "interaction_location_label",
        "Outcome Rates by Interaction Location — Unedited Messages (0% change)",
        sort_by="recipient_responded", min_n=5,
    )

fig, location_outcomes_no_edit = plot_location_outcomes_no_edit()
plt.show()

## Export All Charts

In [ ]:
from datetime import datetime
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for saving

timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
assets_dir = f"/Users/garrettsmith/notebooks/edit analysis/ASSETS {timestamp}"
os.makedirs(assets_dir, exist_ok=True)

def save_fig(fig, name):
    path = os.path.join(assets_dir, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {name}.png")

print(f"Exporting charts to: {assets_dir}\n")

# Standalone charts (each function returns fig or (fig, data))
save_fig(plot_theme_frequency()[0], "01_edit_themes_by_frequency")
save_fig(plot_theme_by_location(), "02_theme_by_interaction_location")
save_fig(plot_action_frequency()[0], "03_edit_action_frequency")
save_fig(plot_actionability_vs_quality()[0], "04_actionability_vs_quality")
save_fig(plot_action_cooccurrence()[0], "05_edit_action_cooccurrence")
save_fig(plot_actions_by_location(), "05b_edit_actions_by_location")
save_fig(plot_repeat_editors()[0], "06_repeat_editor_consistency")

# Outcome rate charts (plot_outcome_rates returns (fig, grouped))
save_fig(plot_outcome_rates(df_combined[df_combined["theme"].notna()], "theme", "Outcome Rates by Edit Theme", sort_by="recipient_responded")[0], "07_outcomes_by_theme")
save_fig(plot_outcome_rates(df_combined[df_combined["actionability_delta"].notna()], "actionability_delta", "Outcome Rates by Actionability Delta")[0], "08_outcomes_by_actionability")
save_fig(plot_outcome_rates(df_combined, "context_strategy", "Outcome Rates by Strategy")[0], "09_outcomes_by_strategy")
save_fig(plot_outcome_rates(df_combined, "context_prompt_variant", "Outcome Rates by Prompt Variant", sort_by="recipient_responded")[0], "10_outcomes_by_prompt_variant")
save_fig(plot_outcome_rates(df_combined[df_combined["context_distance"].notna()], "context_distance", "Outcome Rates by Distance to Recipient")[0], "11_outcomes_by_distance")
save_fig(plot_outcome_rates(df_combined, "context_mutual_bucket", "Outcome Rates by Mutual Connection Count")[0], "12_outcomes_by_mutuals")
save_fig(plot_outcome_rates(df_combined, "context_meeting_lifecycle", "Outcome Rates by Meeting Lifecycle at Send Time")[0], "13_outcomes_by_lifecycle")
save_fig(plot_outcome_rates(df_combined, "pct_changed_bucket", "Outcome Rates by Degree of Editing", preserve_order=True)[0], "14_outcomes_by_edit_degree")
save_fig(plot_outcome_rates(df_combined, "recipient_targeting", "Outcome Rates by Recipient Targeting")[0], "15_outcomes_by_targeting")

# Heatmaps
save_fig(plot_variant_x_theme(), "16_prompt_variant_x_theme_heatmap")
save_fig(plot_strategy_x_theme(), "17_strategy_x_theme_heatmap")
save_fig(plot_distance_x_theme(), "18_distance_x_theme_heatmap")

# Custom outcome charts
save_fig(plot_top_action_outcomes(), "19_outcomes_by_top_edit_action")

# Unedited message charts
df_no_edit = df_combined[df_combined["pct_changed_bucket"] == "0%"]
save_fig(plot_outcome_rates(df_no_edit, "context_prompt_variant", "Outcome Rates by Prompt Variant — Unedited (0% change)", sort_by="recipient_responded", min_n=5)[0], "20_outcomes_by_variant_unedited")
save_fig(plot_location_outcomes_no_edit()[0], "21_outcomes_by_location_unedited")

# Restore inline backend
matplotlib.use("module://matplotlib_inline.backend_inline")

print(f"\nDone! {len(os.listdir(assets_dir))} charts exported to:\n  {assets_dir}")

In [ ]:
df_combined.head(1).columns

In [ ]:
save_fig(plot_outcome_rates(df_combined[df_combined["pct_changed_bucket"] != "0%"], "interaction_location", "Outcome Rates by Interaction Location — Edited Messages", sort_by="recipient_responded", min_n=5)[0], "22_outcomes_by_location_edited")

## Membership Cancellation % by Lifecycle, Last 90 Days

In [ ]:
# Query 1: Cancellation rate by meeting lifecycle
df_lifecycle_churn = query("""
WITH meeting_users AS (
    SELECT DISTINCT MEETING_INITIATOR_USER_ID AS USER_ID, MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING.CONVERSATIONS.CONVERSATION_MEETING_ANALYSES
),
meeting_businesses AS (
    SELECT DISTINCT u.BUSINESS_ID, mu.MEETING_LIFECYCLE
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    INNER JOIN meeting_users mu ON mu.USER_ID = u.ID::VARCHAR
    WHERE u.IS_PRIMARY_USER = TRUE
),
all_meeting_businesses AS (
    SELECT DISTINCT BUSINESS_ID FROM meeting_businesses
),
lifecycle_stats AS (
    SELECT
        CASE mb.MEETING_LIFECYCLE
            WHEN 6 THEN 'completed (6)'
            WHEN 5 THEN 'scheduled (5)'
            WHEN 4 THEN 'mutual_intent (4)'
            WHEN 3 THEN 'not_interested (3)'
            WHEN 2 THEN 'proposed_didnt_work (2)'
            WHEN 1 THEN 'proposed (1)'
        END AS meeting_lifecycle_filter,
        mb.MEETING_LIFECYCLE AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    INNER JOIN meeting_businesses mb ON mb.BUSINESS_ID = s.BUSINESS_ID
    WHERE mb.MEETING_LIFECYCLE IN (1, 2, 3, 4, 5, 6)
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
    GROUP BY 1, 2
),
no_meeting_stats AS (
    SELECT
        'none (no row)' AS meeting_lifecycle_filter,
        -1 AS sort_order,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'churned') AS churned_subscriptions,
        COUNT_IF(s.SUBSCRIPTION_STATUS = 'active') AS active_subscriptions
    FROM ALIGNABLE_REPORTING_V2.MEMBERSHIPS.DIM_MEMBERSHIP_SUBSCRIPTIONS s
    LEFT JOIN all_meeting_businesses amb ON amb.BUSINESS_ID = s.BUSINESS_ID
    WHERE amb.BUSINESS_ID IS NULL
      AND s.ACTIVE_AT >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
),
combined AS (
    SELECT * FROM lifecycle_stats
    UNION ALL
    SELECT * FROM no_meeting_stats
)
SELECT
    meeting_lifecycle_filter,
    churned_subscriptions,
    active_subscriptions,
    churned_subscriptions + active_subscriptions AS total_subscriptions,
    ROUND(churned_subscriptions / NULLIF(churned_subscriptions + active_subscriptions, 0) * 100, 2) AS cancellation_rate_pct
FROM combined
ORDER BY sort_order DESC
""")

# Query 2: Baseline cancellation rate
df_baseline_churn = query("""
SELECT
    churned_subscriptions,
    active_subscriptions,
    churned_subscriptions + active_subscriptions AS total_subscriptions,
    ROUND(churned_subscriptions / NULLIF(churned_subscriptions + active_subscriptions, 0) * 100, 2) AS cancellation_rate_pct
FROM SEMANTIC_VIEW(
    ALIGNABLE_REPORTING_V2.SEMANTIC.SV_MEMBERSHIP_SUBSCRIPTIONS
    METRICS churned_subscriptions, active_subscriptions
    WHERE subscriptions.active_at >= GREATEST('2026-01-20'::DATE, DATEADD(DAY, -90, CURRENT_DATE()))
)
""")

# Combine: add baseline as a row
baseline_row = pd.DataFrame([{
    "MEETING_LIFECYCLE_FILTER": "Baseline",
    "CANCELLATION_RATE_PCT": float(df_baseline_churn["CANCELLATION_RATE_PCT"].iloc[0]),
    "TOTAL_SUBSCRIPTIONS": int(df_baseline_churn["TOTAL_SUBSCRIPTIONS"].iloc[0]),
}])
df_churn = pd.concat([
    df_lifecycle_churn[["MEETING_LIFECYCLE_FILTER", "CANCELLATION_RATE_PCT", "TOTAL_SUBSCRIPTIONS"]],
    baseline_row,
], ignore_index=True)

print(df_churn.to_string(index=False))

In [ ]:
def plot_churn_by_lifecycle():
    x = range(len(df_churn))
    fig, ax = plt.subplots(figsize=(12, 6))

    colors = ["coral" if lbl != "Baseline" else "steelblue" for lbl in df_churn["MEETING_LIFECYCLE_FILTER"]]
    bars = ax.bar(x, df_churn["CANCELLATION_RATE_PCT"].astype(float), color=colors, edgecolor="white")

    ax.set_xticks(list(x))
    ax.set_xticklabels(df_churn["MEETING_LIFECYCLE_FILTER"], rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Cancellation Rate (%)")
    ax.set_title("Membership Cancellation % by Lifecycle, Last 90 Days", pad=25)

    # Percentage + n above each bar
    y_top = ax.get_ylim()[1]
    for i, (bar, rate, n) in enumerate(zip(bars, df_churn["CANCELLATION_RATE_PCT"].astype(float), df_churn["TOTAL_SUBSCRIPTIONS"].astype(int))):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"{rate:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    return fig

fig = plot_churn_by_lifecycle()
plt.show()

## Edit Theme x Outcomes by Sender Business Model

In [ ]:
for biz_model in ["b2b", "b2c", "both", "none"]:
    subset = df_combined[(df_combined["theme"].notna()) & (df_combined["context_sender_business_model"] == biz_model)]
    if len(subset) < 20:
        print(f"Skipping sender={biz_model}: only {len(subset)} rows")
        continue
    fig, _ = plot_outcome_rates(
        subset, "theme",
        f"Edit Theme x Outcomes — Sender Business Model: {biz_model} (n={len(subset):,})",
        sort_by="recipient_responded",
    )
    plt.show()

## Top Edit Actions x Outcomes by Sender Business Model

In [ ]:
for biz_model in ["b2b", "b2c", "both", "none"]:
    subset = df_combined[(df_combined["edit_actions"].notna()) & (df_combined["context_sender_business_model"] == biz_model)].copy()
    if len(subset) < 20:
        print(f"Skipping sender={biz_model}: only {len(subset)} rows")
        continue

    # Compute outcomes for ALL actions with min_n, then take top 10 by recipient_responded
    _ac = Counter()
    for actions in subset["edit_actions"]:
        if isinstance(actions, list):
            for a in actions: _ac[a] += 1

    rows = []
    for action, count in _ac.items():
        if count < 20:
            continue
        mask = subset["edit_actions"].apply(lambda a, act=action: isinstance(a, list) and act in a)
        s = subset[mask]
        rows.append({
            "edit_action": action, "n": len(s),
            "recipient_responded": s["outcome_recipient_responded"].mean() * 100,
            "value_moment": s["outcome_has_value_moment"].mean() * 100,
            "positive_meeting": s["outcome_positive_meeting"].mean() * 100,
            "maybe_met": s["outcome_maybe_met_externally"].mean() * 100,
        })
    if not rows:
        print(f"Skipping sender={biz_model}: no actions with enough data")
        continue
    df_ao = pd.DataFrame(rows).sort_values("recipient_responded", ascending=False).head(10).reset_index(drop=True)

    x = range(len(df_ao)); width = 0.18
    fig, ax = plt.subplots(figsize=(14, 6))
    n_b = 4; offsets = [(i - (n_b-1)/2)*width for i in range(n_b)]
    cols = ["recipient_responded", "value_moment", "positive_meeting", "maybe_met"]
    labels = ["Recipient Responded", "Value Moment", "Met 1:1", "Likely Met"]
    clrs = ["steelblue", "coral", "seagreen", "mediumpurple"]
    for offset, col, label, color in zip(offsets, cols, labels, clrs):
        pos = [i + offset for i in x]; vals = df_ao[col]
        bars = ax.bar(pos, vals, width, label=label, color=color)
        for bar, val in zip(bars, vals):
            if val > 0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f"{val:.1f}%", ha="center", va="bottom", fontsize=5, color="black")
    ax.set_xticks(list(x)); ax.set_xticklabels(df_ao["edit_action"], rotation=45, ha="right", fontsize=8)
    y_top = ax.get_ylim()[1]
    for i, n in enumerate(df_ao["n"]): ax.text(i, y_top*1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")
    ax.set_ylabel("% of edits")
    ax.set_title(f"Top Edit Actions by Response Rate — Sender: {biz_model} (n={len(subset):,})", pad=25)
    ax.legend(loc="upper right", fontsize=7); fig.tight_layout(); plt.subplots_adjust(top=0.90)
    plt.show()

## Edit Theme x Outcomes — Introduce Yourself Locations Only

In [ ]:
introduce_locations = [
    "connection_request_introduce_yourself_llm_modal",
    "daily_action_plan_introduce_yourself_llm",
]

# Debug: check what values actually exist
print("interaction_location unique values in df_combined:")
print(df_combined["interaction_location"].unique())

df_introduce = df_combined[
    (df_combined["theme"].notna())
    & (df_combined["interaction_location"].isin(introduce_locations))
]
print(f"\nIntroduce yourself locations: {len(df_introduce):,} edits")

if len(df_introduce) >= 20:
    fig, _ = plot_outcome_rates(
        df_introduce, "theme",
        f"Edit Theme x Outcomes — Introduce Yourself Locations (n={len(df_introduce):,})",
        sort_by="recipient_responded",
        
    )
    plt.show()

## Top Edit Actions x Outcomes — Introduce Yourself Locations Only

In [ ]:
df_introduce_actions = df_combined[
    (df_combined["edit_actions"].notna())
    & (df_combined["interaction_location"].isin(introduce_locations))
].copy()

# Compute outcomes for ALL actions with min_n, then take top 10 by recipient_responded
_ac = Counter()
for actions in df_introduce_actions["edit_actions"]:
    if isinstance(actions, list):
        for a in actions: _ac[a] += 1

rows = []
for action, count in _ac.items():
    if count < 20:
        continue
    mask = df_introduce_actions["edit_actions"].apply(lambda a, act=action: isinstance(a, list) and act in a)
    s = df_introduce_actions[mask]
    rows.append({
        "edit_action": action, "n": len(s),
        "recipient_responded": s["outcome_recipient_responded"].mean() * 100,
        "value_moment": s["outcome_has_value_moment"].mean() * 100,
        "positive_meeting": s["outcome_positive_meeting"].mean() * 100,
        "maybe_met": s["outcome_maybe_met_externally"].mean() * 100,
    })
df_ao = pd.DataFrame(rows).sort_values("recipient_responded", ascending=False).head(10).reset_index(drop=True)

x = range(len(df_ao)); width = 0.18
fig, ax = plt.subplots(figsize=(14, 6))
n_b = 4; offsets = [(i - (n_b-1)/2)*width for i in range(n_b)]
cols = ["recipient_responded", "value_moment", "positive_meeting", "maybe_met"]
labels = ["Recipient Responded", "Value Moment", "Met 1:1", "Likely Met"]
clrs = ["steelblue", "coral", "seagreen", "mediumpurple"]
for offset, col, label, color in zip(offsets, cols, labels, clrs):
    pos = [i + offset for i in x]; vals = df_ao[col]
    bars = ax.bar(pos, vals, width, label=label, color=color)
    for bar, val in zip(bars, vals):
        if val > 0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f"{val:.1f}%", ha="center", va="bottom", fontsize=5, color="black")
ax.set_xticks(list(x)); ax.set_xticklabels(df_ao["edit_action"], rotation=45, ha="right", fontsize=8)
y_top = ax.get_ylim()[1]
for i, n in enumerate(df_ao["n"]): ax.text(i, y_top*1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")
ax.set_ylabel("% of edits")
ax.set_title(f"Top Edit Actions by Response Rate — Introduce Yourself Locations (n={len(df_introduce_actions):,})", pad=25)
ax.legend(loc="upper right", fontsize=7); fig.tight_layout(); plt.subplots_adjust(top=0.90)
plt.show()

## Sales Pitch Conversion Outcomes by Sender Model x Recipient Type

In [ ]:
THEME = "Converting relationship-building messages into direct sales pitches"

segments = [
    ("B2B → Partner",   "b2b",  "context_recipient_matches_sender_partner_targets"),
    ("B2B → Customer",  "b2b",  "context_recipient_matches_sender_customer_targets"),
    ("B2C → Partner",   "b2c",  "context_recipient_matches_sender_partner_targets"),
    ("B2C → Customer",  "b2c",  "context_recipient_matches_sender_customer_targets"),
    ("Both → Partner",  "both", "context_recipient_matches_sender_partner_targets"),
    ("Both → Customer", "both", "context_recipient_matches_sender_customer_targets"),
]

df_sales = df_combined[df_combined["theme"] == THEME].copy()
print(f"Total rows with theme: {len(df_sales):,}")

rows = []
for label, biz_model, recip_col in segments:
    subset = df_sales[
        (df_sales["context_sender_business_model"] == biz_model)
        & (df_sales[recip_col] == True)
    ]
    if len(subset) == 0:
        continue
    rows.append({
        "segment": label,
        "n": len(subset),
        "recipient_responded": subset["outcome_recipient_responded"].mean() * 100,
        "value_moment": subset["outcome_has_value_moment"].mean() * 100,
        "positive_meeting": subset["outcome_positive_meeting"].mean() * 100,
        "maybe_met": subset["outcome_maybe_met_externally"].mean() * 100,
    })

df_seg = pd.DataFrame(rows)

# Baseline: all unedited messages
df_baseline = df_combined[df_combined["pct_changed_bucket"] == "0%"]
baseline = {
    "segment": "— Baseline (no edits) —",
    "n": len(df_baseline),
    "recipient_responded": df_baseline["outcome_recipient_responded"].mean() * 100,
    "value_moment": df_baseline["outcome_has_value_moment"].mean() * 100,
    "positive_meeting": df_baseline["outcome_positive_meeting"].mean() * 100,
    "maybe_met": df_baseline["outcome_maybe_met_externally"].mean() * 100,
}
df_seg = pd.concat([df_seg, pd.DataFrame([baseline])], ignore_index=True)
print(df_seg.to_string(index=False))
print(f"\nBaseline (no edits): {len(df_baseline):,}")

# --- Chart ---
n_segs = len(df_seg)
x = range(n_segs); width = 0.18
fig, ax = plt.subplots(figsize=(14, 6))
n_b = 4; offsets = [(i - (n_b - 1) / 2) * width for i in range(n_b)]
cols = ["recipient_responded", "value_moment", "positive_meeting", "maybe_met"]
labels = ["Recipient Responded", "Value Moment", "Met 1:1", "Likely Met"]
clrs = ["steelblue", "coral", "seagreen", "mediumpurple"]

for offset, col, lbl, color in zip(offsets, cols, labels, clrs):
    pos = [i + offset for i in x]
    vals = df_seg[col]
    bars = ax.bar(pos, vals, width, label=lbl, color=color)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=7, color="black")

# Dashed line before baseline
ax.axvline(x=n_segs - 1.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xticks(list(x))
ax.set_xticklabels(df_seg["segment"], rotation=30, ha="right", fontsize=9)
tick_labels = ax.get_xticklabels()
tick_labels[-1].set_fontweight("bold")

y_top = ax.get_ylim()[1]
for i, n in enumerate(df_seg["n"]):
    ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=8, color="gray")

ax.set_ylabel("% of edits")
ax.set_title(f"Outcomes When Converting to Sales Pitches\nby Sender Business Model × Recipient Type (n={len(df_sales):,} edited, {len(df_baseline):,} baseline)", pad=25)
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

## Edit Theme Outcomes When Recipient Is a Customer

In [ ]:
df_customer = df_combined[
    (df_combined["context_recipient_matches_sender_customer_targets"] == True)
].copy()

# Baseline: unedited messages sent to customers (0% changed)
df_baseline = df_customer[df_customer["pct_changed_bucket"] == "0%"]
baseline = {
    "theme": "— Baseline (no edits) —",
    "n": len(df_baseline),
    "recipient_responded": df_baseline["outcome_recipient_responded"].mean() * 100,
    "value_moment": df_baseline["outcome_has_value_moment"].mean() * 100,
    "positive_meeting": df_baseline["outcome_positive_meeting"].mean() * 100,
    "maybe_met": df_baseline["outcome_maybe_met_externally"].mean() * 100,
}

# Themed edits sent to customers
df_themed = df_customer[
    (df_customer["theme"].notna())
    & (df_customer["theme"] != "Uncategorized / noise")
]
print(f"Edits where recipient is a customer: {len(df_themed):,}")
print(f"Baseline (no edits, recipient is customer): {len(df_baseline):,}")

theme_counts = df_themed["theme"].value_counts()
min_n = 20
valid_themes = theme_counts[theme_counts >= min_n].index.tolist()

rows = []
for theme in valid_themes:
    subset = df_themed[df_themed["theme"] == theme]
    rows.append({
        "theme": theme,
        "n": len(subset),
        "recipient_responded": subset["outcome_recipient_responded"].mean() * 100,
        "value_moment": subset["outcome_has_value_moment"].mean() * 100,
        "positive_meeting": subset["outcome_positive_meeting"].mean() * 100,
        "maybe_met": subset["outcome_maybe_met_externally"].mean() * 100,
    })

df_theme_cust = pd.DataFrame(rows).sort_values("recipient_responded", ascending=False).reset_index(drop=True)

# Append baseline as last row
df_theme_cust = pd.concat([df_theme_cust, pd.DataFrame([baseline])], ignore_index=True)
print(df_theme_cust.to_string(index=False))

# --- Chart ---
n_themes = len(df_theme_cust)
x = range(n_themes); width = 0.18
fig, ax = plt.subplots(figsize=(16, 7))
n_b = 4; offsets = [(i - (n_b - 1) / 2) * width for i in range(n_b)]
cols = ["recipient_responded", "value_moment", "positive_meeting", "maybe_met"]
labels = ["Recipient Responded", "Value Moment", "Met 1:1", "Likely Met"]
clrs = ["steelblue", "coral", "seagreen", "mediumpurple"]

for offset, col, lbl, color in zip(offsets, cols, labels, clrs):
    pos = [i + offset for i in x]
    vals = df_theme_cust[col]
    bars = ax.bar(pos, vals, width, label=lbl, color=color)
    for bar, val in zip(bars, vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=5, color="black")

# Draw a vertical dashed line before the baseline bar
ax.axvline(x=n_themes - 1.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xticks(list(x))
xlabels = list(df_theme_cust["theme"])
ax.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=7)
# Bold the baseline label
tick_labels = ax.get_xticklabels()
tick_labels[-1].set_fontweight("bold")

y_top = ax.get_ylim()[1]
for i, n in enumerate(df_theme_cust["n"]):
    ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=6, color="gray")

ax.set_ylabel("% of edits")
ax.set_title(f"Edit Theme Outcomes — Recipient Is a Customer (n={len(df_themed):,} edited, {len(df_baseline):,} baseline)", pad=25)
ax.legend(loc="upper right", fontsize=7)
fig.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

## Export: Edit Actions Grouped by Recipient Response Rate

In [ ]:
from collections import Counter
from openpyxl import Workbook
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font, PatternFill
import difflib

def make_rich_diff(original: str, edited: str) -> CellRichText:
    """Word-level diff as openpyxl CellRichText."""
    orig_words = original.split()
    edit_words = edited.split()
    font_normal  = InlineFont(color="000000")
    font_deleted = InlineFont(color="CC0000", strike=True)
    font_added   = InlineFont(color="007700", b=True)
    blocks = []
    matcher = difflib.SequenceMatcher(None, orig_words, edit_words, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            blocks.append(TextBlock(font_normal,  " ".join(orig_words[i1:i2]) + " "))
        elif op == "delete":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
        elif op == "insert":
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))
        elif op == "replace":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))
    return CellRichText(*blocks) if blocks else CellRichText("")

MIN_EXAMPLES = 10
MAX_EXAMPLES_SHOWN = 3

# --- Compute per-action stats, sorted by recipient response rate ---
df_with_actions = df_combined[
    (df_combined["edit_actions"].notna())
    & (df_combined["theme"] != "Uncategorized / noise")
].copy()

_ac = Counter()
for actions in df_with_actions["edit_actions"]:
    if isinstance(actions, list):
        for a in actions:
            _ac[a] += 1

action_stats = []
for action, count in _ac.items():
    if count < MIN_EXAMPLES:
        continue
    mask = df_with_actions["edit_actions"].apply(lambda a, act=action: isinstance(a, list) and act in a)
    s = df_with_actions[mask]
    action_stats.append({
        "edit_action": action,
        "n": len(s),
        "recipient_responded": s["outcome_recipient_responded"].mean() * 100,
        "value_moment": s["outcome_has_value_moment"].mean() * 100,
        "positive_meeting": s["outcome_positive_meeting"].mean() * 100,
        "maybe_met": s["outcome_maybe_met_externally"].mean() * 100,
    })

df_action_stats = pd.DataFrame(action_stats).sort_values("recipient_responded", ascending=False).reset_index(drop=True)
print(f"Edit actions with >= {MIN_EXAMPLES} examples: {len(df_action_stats)}")
print(df_action_stats.head(10).to_string(index=False))

# --- Build the spreadsheet ---
wb = Workbook()
ws = wb.active
ws.title = "Edit Actions by Response Rate"

wrap_top = Alignment(wrap_text=True, vertical="top")
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(bold=True, color="FFFFFF", size=12)
subheader_font = Font(bold=True, size=10)
stat_fill = PatternFill(start_color="D6E4F0", end_color="D6E4F0", fill_type="solid")

# Column widths
ws.column_dimensions["A"].width = 60   # Generated Message
ws.column_dimensions["B"].width = 60   # Sent Message (Diff)
ws.column_dimensions["C"].width = 40   # All Edit Actions
ws.column_dimensions["D"].width = 25   # Theme

row_idx = 1

for _, action_row in df_action_stats.iterrows():
    action = action_row["edit_action"]
    n = int(action_row["n"])
    resp_rate = action_row["recipient_responded"]
    vm_rate = action_row["value_moment"]
    pm_rate = action_row["positive_meeting"]
    mm_rate = action_row["maybe_met"]

    # --- Action header row ---
    cell = ws.cell(row=row_idx, column=1,
                   value=f"{action}  (n={n:,})")
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = wrap_top
    for c in range(2, 5):
        ws.cell(row=row_idx, column=c).fill = header_fill
    ws.merge_cells(start_row=row_idx, start_column=1, end_row=row_idx, end_column=4)

    row_idx += 1

    # --- Stats row ---
    stats_text = (
        f"Response Rate: {resp_rate:.1f}%   |   "
        f"Value Moment: {vm_rate:.1f}%   |   "
        f"Maybe Met: {mm_rate:.1f}%   |   "
        f"Met 1:1: {pm_rate:.1f}%"
    )
    cell = ws.cell(row=row_idx, column=1, value=stats_text)
    cell.font = Font(italic=True, size=9)
    cell.fill = stat_fill
    cell.alignment = wrap_top
    for c in range(2, 5):
        ws.cell(row=row_idx, column=c).fill = stat_fill
    ws.merge_cells(start_row=row_idx, start_column=1, end_row=row_idx, end_column=4)

    row_idx += 1

    # --- Example sub-header ---
    sub_headers = ["Generated Message", "Sent Message (Diff)", "All Edit Actions", "Theme"]
    for c, h in enumerate(sub_headers, 1):
        cell = ws.cell(row=row_idx, column=c, value=h)
        cell.font = subheader_font
        cell.alignment = wrap_top

    row_idx += 1

    # --- Pick examples, sorted: responded first, then non-responded ---
    mask = df_with_actions["edit_actions"].apply(lambda a, act=action: isinstance(a, list) and act in a)
    examples = df_with_actions[mask].sort_values("outcome_recipient_responded", ascending=False).head(MAX_EXAMPLES_SHOWN)

    for _, ex in examples.iterrows():
        ws.cell(row=row_idx, column=1, value=ex.get("generated_message", "")).alignment = wrap_top

        diff_cell = ws.cell(row=row_idx, column=2)
        diff_cell.value = make_rich_diff(
            str(ex.get("generated_message", "") or ""),
            str(ex.get("sent_message", "") or ""),
        )
        diff_cell.alignment = wrap_top

        actions_str = ", ".join(ex["edit_actions"]) if isinstance(ex["edit_actions"], list) else str(ex["edit_actions"])
        ws.cell(row=row_idx, column=3, value=actions_str).alignment = wrap_top
        ws.cell(row=row_idx, column=4, value=ex.get("theme", "")).alignment = wrap_top

        row_idx += 1

    # Blank separator row
    row_idx += 1

output_path = "/Users/garrettsmith/notebooks/edit analysis/edit_actions_by_response_rate.xlsx"
wb.save(output_path)
print(f"\nSaved → {output_path}")
print(f"Total rows written: {row_idx - 1}")

In [ ]:
import math

nan_count = sum(1 for theme in df_combined["theme"] if isinstance(theme, float) and math.isnan(theme))
print(f"Rows where theme is NaN: {nan_count:,} / {len(df_combined):,} ({nan_count/len(df_combined)*100:.1f}%)")

In [ ]:
themes_of_interest = [
    "Humanizing outreach from sales pitch to genuine connection",
    "Replacing vague meeting interest with frictionless scheduling mechanics",
    "Professionalizing tone while tightening overly casual AI language",
]

df_theme_subset = df_edits[df_edits["theme"].isin(themes_of_interest)]
df_sampled = df_theme_subset.groupby("theme", group_keys=False).apply(lambda g: g.sample(100, random_state=42))

print(df_sampled["theme"].value_counts())

In [ ]:
df_sampled[["GENERATED_MESSAGE", "SENT_MESSAGE", "theme"]]

In [ ]:
import os

output_dir = "/Users/garrettsmith/notebooks/edit analysis/examples for prompt updates"

for theme, group in df_sampled.groupby("theme"):
    # Create a short filename from the theme
    if "Humanizing" in theme:
        fname = "humanizing_outreach.csv"
    elif "scheduling" in theme:
        fname = "scheduling_mechanics.csv"
    elif "Professionalizing" in theme:
        fname = "professionalizing_tone.csv"
    
    group[["GENERATED_MESSAGE", "SENT_MESSAGE"]].to_csv(os.path.join(output_dir, fname), index=False)
    print(f"Wrote {len(group)} rows to {fname}")

In [ ]:
top3_themes = [
    "Humanizing outreach from sales pitch to genuine connection",
    "Replacing vague meeting interest with frictionless scheduling mechanics",
    "Professionalizing tone while tightening overly casual AI language",
]

df_top3 = df_combined[df_combined["theme"].apply(lambda t: any(str(t).startswith(prefix) for prefix in top3_themes))].copy()
df_top3["theme_short"] = df_top3["theme"].apply(
    lambda t: next((v for k, v in {
        "Humanizing outreach from sales pitch to genuine connection": "Humanizing outreach",
        "Replacing vague meeting interest with frictionless scheduling mechanics": "Frictionless scheduling",
        "Professionalizing tone while tightening overly casual AI language": "Professionalizing tone",
    }.items() if str(t).startswith(k)), t)
)

df_baseline = df_combined[df_combined["pct_changed_bucket"] == "0%"]

outcome_cols = ["outcome_recipient_responded", "outcome_has_value_moment", "outcome_maybe_met_externally", "outcome_positive_meeting"]
bar_labels = ["Recipient Responded", "Value Moment", "Likely Met", "Met 1:1"]
bar_colors = ["goldenrod", "coral", "seagreen", "mediumpurple"]
resp_idx = 0  # index of outcome_recipient_responded in outcome_cols

# Build theme groups, then sort by recipient_responded
theme_groups = []
for theme in ["Humanizing outreach", "Frictionless scheduling", "Professionalizing tone"]:
    subset = df_top3[df_top3["theme_short"] == theme]
    theme_groups.append((theme, len(subset), [subset[c].mean() * 100 for c in outcome_cols]))
theme_groups.sort(key=lambda g: g[2][resp_idx], reverse=True)

# Build baseline groups, then sort by recipient_responded
baseline_groups = [
    (f"Unedited (0% changed)", len(df_baseline), [df_baseline[c].mean() * 100 for c in outcome_cols]),
    (f"All Messages", len(df_combined), [df_combined[c].mean() * 100 for c in outcome_cols]),
]
baseline_groups.sort(key=lambda g: g[2][resp_idx], reverse=True)

groups = theme_groups + baseline_groups
group_names = [g[0] for g in groups]
group_ns = [g[1] for g in groups]
group_rates = [g[2] for g in groups]

n_groups = len(groups)
n_bars = len(outcome_cols)
width = 0.18
fig, ax = plt.subplots(figsize=(14, 6))

for j, (label, color) in enumerate(zip(bar_labels, bar_colors)):
    offset = (j - (n_bars - 1) / 2) * width
    positions = [i + offset for i in range(n_groups)]
    values = [group_rates[i][j] for i in range(n_groups)]
    bars = ax.bar(positions, values, width, label=label, color=color)
    for bar in bars:
        val = bar.get_height()
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=7)

# Dotted separator between themes and baselines
ax.axvline(x=len(theme_groups) - 0.5, color="gray", linestyle=":", linewidth=1.5)

ax.set_xticks(range(n_groups))
ax.set_xticklabels(group_names, rotation=30, ha="right", fontsize=9)

y_top = ax.get_ylim()[1]
for i, n in enumerate(group_ns):
    ax.text(i, y_top * 1.02, f"n={n:,}", ha="center", va="bottom", fontsize=7, color="gray")

ax.set_ylabel("% of messages")
ax.set_title("Edit Theme x Outcomes, Top 3 Edit Themes", pad=25)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.show()

In [ ]:
df_sales_pitch_b2b = df_combined[
    (df_combined["theme"].apply(lambda t: str(t).startswith("Converting relationship-building messages into direct sales pitches")))
    & (df_combined["context_sender_business_model"] == "b2b")
].copy()

print(f"Rows: {len(df_sales_pitch_b2b):,}")
df_sales_pitch_b2b

In [ ]:
# Get business descriptions for source and target users
user_ids = pd.concat([df_sales_pitch_b2b["source_user_id"], df_sales_pitch_b2b["target_user_id"]]).unique().tolist()

df_user_biz = query(f"""
    SELECT u.ID::STRING AS user_id, b.DESCRIPTION AS business_description
    FROM ALIGNABLE_REPORTING_V2.CORE.DIM_USER u
    JOIN ALIGNABLE_REPORTING.BUSINESSES.BUSINESSES b ON b.ID = u.BUSINESS_ID
    WHERE u.ID IN ({','.join(str(int(float(uid))) for uid in user_ids if pd.notna(uid))})
      AND u.IS_PRIMARY_USER = TRUE
""")

biz_desc_map = df_user_biz.set_index("USER_ID")["BUSINESS_DESCRIPTION"].to_dict()

# Ensure user IDs are the same type (string) for mapping
df_sales_pitch_b2b_detail = df_sales_pitch_b2b[[
    "source_user_id", "target_user_id", "generated_message", "sent_message",
    "context_strategy", "interaction_location", "recipient_targeting"
]].copy()

df_sales_pitch_b2b_detail.insert(1, "source_user_description", df_sales_pitch_b2b_detail["source_user_id"].astype(str).map(biz_desc_map))
df_sales_pitch_b2b_detail.insert(3, "target_user_description", df_sales_pitch_b2b_detail["target_user_id"].astype(str).map(biz_desc_map))
df_sales_pitch_b2b_detail = df_sales_pitch_b2b_detail.rename(columns={"context_strategy": "strategy"})

print(f"Rows: {len(df_sales_pitch_b2b_detail):,}")
print(f"source_user_description NaN: {df_sales_pitch_b2b_detail['source_user_description'].isna().sum()}")
print(f"target_user_description NaN: {df_sales_pitch_b2b_detail['target_user_description'].isna().sum()}")
df_sales_pitch_b2b_detail

In [ ]:
from openpyxl import Workbook
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font
import difflib

def make_rich_diff(original: str, edited: str) -> CellRichText:
    """Word-level diff as openpyxl CellRichText.
    Deletions = red strikethrough, insertions = green bold, unchanged = black."""
    orig_words = (original or "").split()
    edit_words = (edited or "").split()
    font_normal  = InlineFont(color="000000")
    font_deleted = InlineFont(color="CC0000", strike=True)
    font_added   = InlineFont(color="007700", b=True)
    blocks = []
    matcher = difflib.SequenceMatcher(None, orig_words, edit_words, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "equal":
            blocks.append(TextBlock(font_normal,  " ".join(orig_words[i1:i2]) + " "))
        elif op == "delete":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
        elif op == "insert":
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))
        elif op == "replace":
            blocks.append(TextBlock(font_deleted, " ".join(orig_words[i1:i2]) + " "))
            blocks.append(TextBlock(font_added,   " ".join(edit_words[j1:j2]) + " "))
    return CellRichText(*blocks) if blocks else CellRichText("")

wb = Workbook()
ws = wb.active
ws.title = "B2B Sales Pitch Edits"

headers = ["Source User ID", "Source User Description", "Target User ID", "Target User Description",
           "Generated Message", "Sent Message", "Sent Message (Annotated)", "Strategy",
           "Interaction Location", "Recipient Targeting"]
for col, header in enumerate(headers, 1):
    cell = ws.cell(row=1, column=col, value=header)
    cell.font = Font(bold=True)

widths = [15, 40, 15, 40, 60, 60, 60, 25, 30, 20]
for col, w in enumerate(widths, 1):
    ws.column_dimensions[chr(64 + col) if col <= 26 else "A"].width = w

wrap_top = Alignment(wrap_text=True, vertical="top")

for row_idx, row in enumerate(df_sales_pitch_b2b_detail.itertuples(), start=2):
    ws.cell(row=row_idx, column=1, value=row.source_user_id).alignment = wrap_top
    ws.cell(row=row_idx, column=2, value=row.source_user_description).alignment = wrap_top
    ws.cell(row=row_idx, column=3, value=row.target_user_id).alignment = wrap_top
    ws.cell(row=row_idx, column=4, value=row.target_user_description).alignment = wrap_top
    ws.cell(row=row_idx, column=5, value=row.generated_message).alignment = wrap_top
    ws.cell(row=row_idx, column=6, value=row.sent_message).alignment = wrap_top
    diff_cell = ws.cell(row=row_idx, column=7)
    diff_cell.value = make_rich_diff(row.generated_message or "", row.sent_message or "")
    diff_cell.alignment = wrap_top
    ws.cell(row=row_idx, column=8, value=row.strategy).alignment = wrap_top
    ws.cell(row=row_idx, column=9, value=row.interaction_location).alignment = wrap_top
    ws.cell(row=row_idx, column=10, value=row.recipient_targeting).alignment = wrap_top

output_path = "/Users/garrettsmith/notebooks/edit analysis/sales_pitch_b2b_edits.xlsx"
wb.save(output_path)
print(f"Exported {len(df_sales_pitch_b2b_detail):,} rows to {output_path}")